# Olist Regional and Customer Performance Analysis

## Business Objective

This analysis evaluates how delivered-order performance varies across Brazilian states and compares one-time customers with repeat customers. Regional performance is assessed using order volume, item volume, GMV, freight burden, delivery performance, and review outcomes. Customer-level analysis measures purchasing frequency and commercial contribution while acknowledging that customer classifications are limited to the available dataset period.

## 1. Load and Validate the Analysis Scope

The analysis uses the validated delivered-order-item dataset created during data preparation. Initial checks confirm that the dataset retains the expected delivered-order population and contains the customer, regional, commercial, delivery, and review fields required for the analysis.

In [1]:
# Import required libraries and define the processed-data path
import numpy as np
import pandas as pd 
from pathlib import Path

processed_data_path = Path("../data/processed")

print(processed_data_path.resolve())


C:\Users\Hp\DataAnalyst\data\processed


In [2]:
# Load the analysis-ready delivered-order-item dataset for regional and customer analysis.
delivered_order_items = pd.read_parquet(processed_data_path / "delivered_order_items_analysis_ready.parquet")

In [3]:
delivered_order_items.shape

(110197, 74)

### 1.1 Validate the Delivered-Item Analysis Scope

The loaded dataset is validated at both item and order levels before regional and customer aggregation. The checks confirm the expected delivered-item population, uniqueness of the order-item key, availability of customer identifiers and state information, and consistency with the previously validated Day 5 totals.

In [4]:
# Validate the delivered-item scope and required customer and regional identifiers.
print("total number of delivered order items:", delivered_order_items.shape[0])
print("total number of unique delivered orders:", delivered_order_items["order_id"].nunique())
print("Duplicate order_id and order_item_id combinations:", delivered_order_items[['order_id', 'order_item_id']].duplicated().sum())
print("Distinct customer_unique_id values:", delivered_order_items["customer_unique_id"].nunique())
print("Distinct customer_state values:",delivered_order_items['customer_state'].nunique())
required_identifier_columns = ["order_id", "order_item_id", 'customer_id',"customer_unique_id", "customer_state"]
print("Missing values in the five required identifier columns:","\n",delivered_order_items[required_identifier_columns].isna().sum())
print("Unique order statuses:", delivered_order_items["order_status"].unique().tolist())

total number of delivered order items: 110197
total number of unique delivered orders: 96478
Duplicate order_id and order_item_id combinations: 0
Distinct customer_unique_id values: 93358
Distinct customer_state values: 27
Missing values in the five required identifier columns: 
 order_id              0
order_item_id         0
customer_id           0
customer_unique_id    0
customer_state        0
dtype: int64
Unique order statuses: ['delivered']


In [5]:
# Validate order, customer, and state mapping consistency before regional aggregation.
print("Orders mapping to more than one customer_id:", delivered_order_items.groupby('order_id')['customer_id'].nunique().gt(1).sum())
print("Orders mapping to more than one customer_unique_id:", delivered_order_items.groupby('order_id')['customer_unique_id'].nunique().gt(1).sum())
print("Orders mapping to more than one customer_state:", delivered_order_items.groupby('order_id')['customer_state'].nunique().gt(1).sum())
print("customer_id values mapping to more than one customer_unique_id:", delivered_order_items.groupby('customer_id')['customer_unique_id'].nunique().gt(1).sum())
print("customer_id values mapping to more than one customer_state:", delivered_order_items.groupby('customer_id')['customer_state'].nunique().gt(1).sum())
print("Number of customer_unique_id values associated with more than one state:", delivered_order_items.groupby('customer_unique_id')['customer_state'].nunique().gt(1).sum())

Orders mapping to more than one customer_id: 0
Orders mapping to more than one customer_unique_id: 0
Orders mapping to more than one customer_state: 0
customer_id values mapping to more than one customer_unique_id: 0
customer_id values mapping to more than one customer_state: 0
Number of customer_unique_id values associated with more than one state: 37


### Mapping Validation Summary

All delivered orders map consistently to one customer identifier, one unique customer, and one customer state. Customer-level identifiers also map consistently to their corresponding unique-customer identifiers and states. However, 37 unique customers have delivered orders associated with more than one state, which may represent relocation or the use of different delivery locations. Regional metrics therefore use the state recorded for each individual order. Customer-level regional attribution will use the state associated with the customer’s most recent delivered order.

## 2. Create the Regional-Performance Table

Regional performance requires two analytical grains. Item-level records are used for delivered-item count, GMV, and freight value, while one row per delivered order is used for order count, distinct customers, delivery performance, and review measures. Separating these grains prevents order-level values from being duplicated across multi-item orders.

### 2.1 Create the Regional Order-Level Scope

In [6]:
# Create and validate one row per delivered order for regional order-level measures.
regional_order_scope = delivered_order_items.drop_duplicates(subset=["order_id"]).copy()

In [7]:
# Validate the regional order-level scope and required customer and regional identifiers.
print("regional order scope shape:", regional_order_scope.shape)
print("Duplicate order_id count:", regional_order_scope["order_id"].duplicated().sum())
print("Distinct order_id count:", regional_order_scope["order_id"].nunique())
print("Distinct states:", regional_order_scope["customer_state"].nunique())
print("Missing customer_state values:", regional_order_scope['customer_state'].isna().sum())
print("Distinct customer_unique_id values:", regional_order_scope["customer_unique_id"].nunique())

regional order scope shape: (96478, 74)
Duplicate order_id count: 0
Distinct order_id count: 96478
Distinct states: 27
Missing customer_state values: 0
Distinct customer_unique_id values: 93358


In [8]:
# Inspect available columns required for regional-performance measures.
column_names_containing_customer = regional_order_scope.filter(like='customer').columns.to_list()
print("column name containing 'customer':", column_names_containing_customer)
print("\n")
column_names_containing_price = regional_order_scope.filter(like='price').columns.to_list()
print("column name containing 'price':", column_names_containing_price)
print("\n")
column_names_containing_freight = regional_order_scope.filter(like='freight').columns.to_list()
print("column name containing 'freight':", column_names_containing_freight)
print("\n")
column_names_containing_delivery = regional_order_scope.filter(like='delivery').columns.to_list()
print("column name containing 'delivery':", column_names_containing_delivery)
print("\n")
column_names_containing_review = regional_order_scope.filter(like='review').columns.to_list()
print("column name containing 'review':", column_names_containing_review)
print("\n")
column_names_containing_order_item = regional_order_scope.filter(like='order_item').columns.to_list()
print("column name containing 'order_item':", column_names_containing_order_item)

column name containing 'customer': ['customer_id', 'order_delivered_customer_date', 'flag_delivered_missing_customer_date', 'flag_customer_before_carrier', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


column name containing 'price': ['price']


column name containing 'freight': ['freight_value', 'freight_burden_percentage']


column name containing 'delivery': ['order_estimated_delivery_date', 'delivery_duration_days', 'delivery_difference_days', 'delivery_calendar_difference_days', 'delivery_status']


column name containing 'review': ['review_record_count', 'average_review_score', 'review_title_count', 'review_message_count', 'average_review_response_duration', 'flag_review_response_over_30_days', 'flag_missing_review_record']


column name containing 'order_item': ['order_item_id']


In [9]:
# Validate the item-level and order-level fields required for regional metrics.

# item-level required for regional metrics.
item_level_columns = ['order_item_id', 'customer_state', 'price', 'freight_value']

# order-level required for regional metrics
order_level_columns = ['order_id', 'customer_unique_id', 'customer_state', 'delivery_duration_days', 'delivery_status', 'average_review_score', 'flag_missing_review_record']

item_level_columns_missing_values_count = delivered_order_items[item_level_columns].isna().sum()
print("item level missing values count:","\n",item_level_columns_missing_values_count)
print("\n")

order_level_columns_missing_values_count = regional_order_scope[order_level_columns].isna().sum()
print("order level missing values count:","\n",order_level_columns_missing_values_count)
print("\n")

print("Delivery-status value counts:","\n", regional_order_scope['delivery_status'].value_counts(dropna=False))


item level missing values count: 
 order_item_id     0
customer_state    0
price             0
freight_value     0
dtype: int64


order level missing values count: 
 order_id                        0
customer_unique_id              0
customer_state                  0
delivery_duration_days          8
delivery_status                 0
average_review_score          646
flag_missing_review_record      0
dtype: int64


Delivery-status value counts: 
 delivery_status
early      88644
late        6534
on_time     1292
missing        8
Name: count, dtype: int64


In [10]:
print("Minimum and maximum average_review_score","\n"
      "Minimum average_review_score:", regional_order_scope['average_review_score'].min(),"\n"
      "Maximum average_review_score:", regional_order_scope['average_review_score'].max())
print("\n")

regional_review_delivery_flag_columns = ['flag_delivered_missing_customer_date', 'flag_customer_before_carrier', 'flag_review_response_over_30_days', 'flag_missing_review_record']

print("Review and delivery flag missing-value counts:","\n",regional_order_scope[regional_review_delivery_flag_columns].isna().sum())
print("\n")

print("regional review delivery flags data type: ", regional_order_scope[regional_review_delivery_flag_columns].dtypes)

Minimum and maximum average_review_score 
Minimum average_review_score: 1.0 
Maximum average_review_score: 5.0


Review and delivery flag missing-value counts: 
 flag_delivered_missing_customer_date      0
flag_customer_before_carrier              0
flag_review_response_over_30_days       646
flag_missing_review_record                0
dtype: int64


regional review delivery flags data type:  flag_delivered_missing_customer_date       bool
flag_customer_before_carrier               bool
flag_review_response_over_30_days       boolean
flag_missing_review_record                 bool
dtype: object


### 2.2 Aggregate Regional Item-Level Measures

Item-level records are grouped by customer state to calculate delivered-item volume, merchandise value, and freight value. These additive measures remain at item grain so that every purchased item and its corresponding commercial values are included.


In [11]:
# Aggregate delivered-item volume, GMV, and freight value by customer state.
regional_item_performance = delivered_order_items.groupby('customer_state').agg(regional_item_count= ('order_item_id', 'count'),
                                                                                regional_gmv= ('price', 'sum'),
                                                                                regional_freight_value= ('freight_value', 'sum')
                                                                                ).reset_index()

regional_item_performance

,customer_state,regional_item_count,regional_gmv,regional_freight_value
0,AC,91,15930.97,3644.36
1,AL,427,78855.72,15316.77
2,AM,163,22155.84,5429.63
3,AP,81,13374.81,2767.00
4,BA,3683,493584.14,97553.67
5,CE,1426,219757.38,46679.39
6,DF,2355,296498.41,49624.94
7,ES,2225,268643.45,49014.48
8,GO,2277,282836.70,51375.65
9,MA,800,117009.38,30794.17


In [12]:
# Validate regional_item_performance delivered-item volume, GMV, and freight value by customer state.
print("regional_item_performance shape:", regional_item_performance.shape)
print("Duplicate state values:", regional_item_performance['customer_state'].duplicated().sum())
print("Missing values: ","\n", regional_item_performance.isna().sum())
print("Item total: ", regional_item_performance['regional_item_count'].sum())
print("GMV total: ", regional_item_performance['regional_gmv'].sum())
print("Freight total: ", regional_item_performance['regional_freight_value'].sum())

regional_item_performance shape: (27, 4)
Duplicate state values: 0
Missing values:  
 customer_state            0
regional_item_count       0
regional_gmv              0
regional_freight_value    0
dtype: int64
Item total:  110197
GMV total:  13221498.110000001
Freight total:  2198275.64


### 2.3 Aggregate Regional Order-Level Measures

The one-row-per-order scope is grouped by customer state to calculate delivered-order volume, customer reach, delivery performance, and review outcomes without duplicating order-level information across purchased items. Valid-denominator counts are retained so that regional delivery and review rates can be calculated accurately.

In [13]:
# Create order-level eligibility and outcome flags for regional delivery and review measures.
regional_order_scope['valid_delivery_order'] = regional_order_scope['delivery_status'] != 'missing'
regional_order_scope['late_delivery_order'] = regional_order_scope['delivery_status'] == 'late'
regional_order_scope['reviewed_order'] = regional_order_scope['flag_missing_review_record'] == False

In [14]:
regional_order_performance = regional_order_scope.groupby('customer_state').agg(regional_order_count=('order_id', 'count'),
                                                                                regional_customer_count= ('customer_unique_id', 'nunique'),
                                                                                regional_average_delivery_duration_days= ('delivery_duration_days', 'mean'),
                                                                                regional_valid_delivery_order_count= ('valid_delivery_order', 'sum'),
                                                                                regional_late_order_count= ('late_delivery_order', 'sum'),
                                                                                regional_reviewed_order_count= ('reviewed_order', 'sum'),
                                                                                regional_average_review_score= ('average_review_score', 'mean'),
                                                                                regional_review_response_over_30_days_order_count= ('flag_review_response_over_30_days', 'sum')
                                                                                ).reset_index()
regional_order_performance.shape

(27, 9)

In [15]:
# Validate regional order-level measures and their valid-denominator totals.
print("Duplicate state: ", regional_order_performance['customer_state'].duplicated().sum())
print("Missing values across the table: ", regional_order_performance.isna().sum().sum())
print("Sum of regional_order_count: ", regional_order_performance['regional_order_count'].sum())
print("Sum of regional_valid_delivery_order_count: ", regional_order_performance['regional_valid_delivery_order_count'].sum())
print("Sum of regional_late_order_count: ", regional_order_performance['regional_late_order_count'].sum())
print("Sum of regional_reviewed_order_count: ",regional_order_performance['regional_reviewed_order_count'].sum())
print("Sum of response-over-30-days orders: ", regional_order_performance['regional_review_response_over_30_days_order_count'].sum())
print("Minimum and maximum average delivery duration","\n",
      "Minimum:",regional_order_performance['regional_average_delivery_duration_days'].min(), "\n","Maximum:",regional_order_performance['regional_average_delivery_duration_days'].max())
print("Minimum and maximum average review score","\n",
      "Minimum:",regional_order_performance['regional_average_review_score'].min(), "\n","Maximum:",regional_order_performance['regional_average_review_score'].max())


Duplicate state:  0
Missing values across the table:  0
Sum of regional_order_count:  96478
Sum of regional_valid_delivery_order_count:  96470
Sum of regional_late_order_count:  6534
Sum of regional_reviewed_order_count:  95832
Sum of response-over-30-days orders:  645
Minimum and maximum average delivery duration 
 Minimum: 8.761385996084925 
 Maximum: 29.387546014001806
Minimum and maximum average review score 
 Minimum: 3.833567415730337 
 Maximum: 4.246550625315555


### 2.4 Combine Regional Item-Level and Order-Level Measures

The validated item-level and order-level state summaries are combined using customer state as the common key. A one-to-one merge is required because each component table contains exactly one row per state. The combined table forms the foundation for regional contribution, average-value, delivery-rate, and review-rate calculations.

In [16]:
# Combine the item-level and order-level regional performance measures.
regional_performance = pd.merge(regional_item_performance, regional_order_performance, on='customer_state', how='outer', validate='one_to_one') 

In [17]:
# Validate regional_performace 
print("regional_performance shape: ", regional_performance.shape)
print("Duplicate state count: ", regional_performance['customer_state'].duplicated().sum())
print("Missing value: ", regional_performance.isna().sum().sum())
print("Distinct state: ", regional_performance['customer_state'].nunique())

regional_performance shape:  (27, 12)
Duplicate state count:  0
Missing value:  0
Distinct state:  27


### 2.5 Calculate Derived Regional Performance Metrics

Derived regional metrics measure each state’s contribution to total activity, typical order value and size, freight burden, delivery reliability, and review coverage. Rates use their appropriate valid denominators so that unavailable delivery or review outcomes are not treated as successful or unsuccessful outcomes.

In [18]:
# Calculate regional contribution, order-value, freight, delivery, and review metrics.
regional_performance['regional_average_order_value'] = regional_performance['regional_gmv']/regional_performance['regional_order_count']
regional_performance['regional_average_items_per_order'] = regional_performance['regional_item_count']/regional_performance['regional_order_count']
regional_performance['regional_order_share_percentage'] = (regional_performance['regional_order_count']/regional_performance['regional_order_count'].sum()) *100
regional_performance['regional_item_share_percentage'] = (regional_performance['regional_item_count']/regional_performance['regional_item_count'].sum())*100
regional_performance['regional_gmv_share_percentage'] = (regional_performance['regional_gmv']/regional_performance['regional_gmv'].sum())*100
regional_performance['regional_freight_burden_percentage'] = (regional_performance['regional_freight_value']/regional_performance['regional_gmv'])*100
regional_performance['regional_late_delivery_rate'] = (regional_performance['regional_late_order_count']/regional_performance['regional_valid_delivery_order_count'])*100
regional_performance['regional_review_coverage_rate'] = (regional_performance['regional_reviewed_order_count']/regional_performance['regional_order_count'])*100
regional_performance['regional_review_response_over_30_days_rate'] = (regional_performance['regional_review_response_over_30_days_order_count']/regional_performance['regional_reviewed_order_count'])*100


In [19]:
# Validate the derived regional-performance metrics and contribution shares.
regional_derived_metric_columns = [
    'regional_average_order_value',
    'regional_average_items_per_order',
    'regional_order_share_percentage',
    'regional_item_share_percentage',
    'regional_gmv_share_percentage',
    'regional_freight_burden_percentage',
    'regional_late_delivery_rate',
    'regional_review_coverage_rate',
    'regional_review_response_over_30_days_rate']

print("regional_performance shape: ", regional_performance.shape)
print("regional_derived_metric_columns data types: ","\n", regional_performance[regional_derived_metric_columns].dtypes)
print("Missing-value counts: ", regional_performance[regional_derived_metric_columns].isna().sum().sum())
print("infinite value counts: ", np.isinf(regional_performance[regional_derived_metric_columns]).sum().sum())
print("Order shares sum to approximately 100%: ", np.isclose(regional_performance['regional_order_share_percentage'].sum(), 100))
print("Item shares sum to approximately 100%: ", np.isclose(regional_performance['regional_item_share_percentage'].sum(), 100))
print("GMV shares sum to approximately 100%: ", np.isclose(regional_performance['regional_gmv_share_percentage'].sum(), 100))

regional_performance shape:  (27, 21)
regional_derived_metric_columns data types:  
 regional_average_order_value                  float64
regional_average_items_per_order              float64
regional_order_share_percentage               float64
regional_item_share_percentage                float64
regional_gmv_share_percentage                 float64
regional_freight_burden_percentage            float64
regional_late_delivery_rate                   float64
regional_review_coverage_rate                 float64
regional_review_response_over_30_days_rate    Float64
dtype: object
Missing-value counts:  0
infinite value counts:  0
Order shares sum to approximately 100%:  True
Item shares sum to approximately 100%:  True
GMV shares sum to approximately 100%:  True


In [20]:
print("Minimum and maximum values for all nine derived metrics")
print("regional_average_order_value","\n",
      "Minimum: ", regional_performance['regional_average_order_value'].min(), "\n", 
      "Maximum: ", regional_performance['regional_average_order_value'].max())
print("\n")

print("regional_average_items_per_order","\n",
      "Minimum: ", regional_performance['regional_average_items_per_order'].min(), "\n", 
      "Maximum: ", regional_performance['regional_average_items_per_order'].max())
print("\n")

print("regional_order_share_percentage","\n",
      "Minimum: ", regional_performance['regional_order_share_percentage'].min(), "\n", 
      "Maximum: ", regional_performance['regional_order_share_percentage'].max())
print("\n")

print("regional_item_share_percentage","\n",
      "Minimum: ", regional_performance['regional_item_share_percentage'].min(), "\n", 
      "Maximum: ", regional_performance['regional_item_share_percentage'].max())
print("\n")

print("regional_gmv_share_percentage","\n",
      "Minimum: ", regional_performance['regional_gmv_share_percentage'].min(), "\n", 
      "Maximum: ", regional_performance['regional_gmv_share_percentage'].max())


Minimum and maximum values for all nine derived metrics
regional_average_order_value 
 Minimum:  125.12365521838967 
 Maximum:  217.76947775628628


regional_average_items_per_order 
 Minimum:  1.0755667506297228 
 Maximum:  1.208955223880597


regional_order_share_percentage 
 Minimum:  0.042496735006944585 
 Maximum:  41.979518646738114


regional_item_share_percentage 
 Minimum:  0.0417434231421908 
 Maximum:  42.149967784966925


regional_gmv_share_percentage 
 Minimum:  0.05337874680526653 
 Maximum:  38.328736409735036


In [21]:
print("regional_freight_burden_percentage","\n",
      "Minimum: ", regional_performance['regional_freight_burden_percentage'].min(), "\n", 
      "Maximum: ", regional_performance['regional_freight_burden_percentage'].max())
print("\n")

print("regional_late_delivery_rate","\n",
      "Minimum: ", regional_performance['regional_late_delivery_rate'].min(), "\n", 
      "Maximum: ", regional_performance['regional_late_delivery_rate'].max())
print("\n")

print("regional_review_coverage_rate","\n",
      "Minimum: ", regional_performance['regional_review_coverage_rate'].min(), "\n", 
      "Maximum: ", regional_performance['regional_review_coverage_rate'].max())
print("\n")

print("regional_review_response_over_30_days_rate","\n",
      "Minimum: ", regional_performance['regional_review_response_over_30_days_rate'].min(), "\n", 
      "Maximum: ", regional_performance['regional_review_response_over_30_days_rate'].max())



regional_freight_burden_percentage 
 Minimum:  13.854001815711536 
 Maximum:  28.08442685551621


regional_late_delivery_rate 
 Minimum:  2.7586206896551726 
 Maximum:  21.41057934508816


regional_review_coverage_rate 
 Minimum:  98.50746268656717 
 Maximum:  100.0


regional_review_response_over_30_days_rate 
 Minimum:  0.0 
 Maximum:  1.5151515151515151


### Regional-Performance Table Validation Summary

The regional-performance table contains all 27 customer states and reconciles fully with the validated delivered-order-item dataset. State order, item, and GMV contribution shares each total approximately 100%, while the derived commercial, freight, delivery, and review measures contain no missing or infinite values. The table is therefore suitable for regional ranking and performance screening.

## 3. Evaluate Regional Performance and Identify Low-Performance State Candidates

States are evaluated using delivered-order volume, item volume, GMV contribution, customer reach, average order value, freight burden, delivery reliability, and review outcomes. Order-count and GMV distributions are examined before defining low performance so that screening thresholds are based on observed state performance. The resulting classification is an analytical screening tool and does not account for population, market coverage, logistics infrastructure, profitability, or strategic importance.

In [22]:
# Rank states by delivered-order volume and GMV.

regional_performance['regional_order_count_rank'] = regional_performance['regional_order_count'].rank(method='dense', ascending= False)
regional_performance['regional_gmv_rank'] = regional_performance['regional_gmv'].rank(method='dense', ascending=False)
regional_performance[['customer_state', 'regional_order_count', 'regional_order_count_rank', 'regional_gmv', 'regional_gmv_rank']].sort_values('regional_order_count_rank')


,customer_state,regional_order_count,regional_order_count_rank,regional_gmv,regional_gmv_rank
25,SP,40501,1.0,5067633.16,1.0
18,RJ,12350,2.0,1759651.13,2.0
10,MG,11354,3.0,1552481.83,3.0
22,RS,5345,4.0,728897.47,4.0
17,PR,4923,5.0,666063.51,5.0
23,SC,3546,6.0,507012.13,6.0
4,BA,3256,7.0,493584.14,7.0
6,DF,2080,8.0,296498.41,8.0
7,ES,1995,9.0,268643.45,10.0
8,GO,1957,10.0,282836.70,9.0


In [23]:
# Validate regional delivered-order and GMV rankings.
print("Missing values in both rank columns:","\n", regional_performance[['regional_order_count_rank','regional_gmv_rank']].isna().sum())
print("Minimum order-count rank:", regional_performance['regional_order_count_rank'].min())
print("Minimum GMV rank:", regional_performance['regional_gmv_rank'].min())
print("Maximum order-count rank:", regional_performance['regional_order_count_rank'].max())
print("Maximum GMV rank:", regional_performance['regional_gmv_rank'].max())
print("Data types of both rank columns:","\n",regional_performance[['regional_order_count_rank','regional_gmv_rank']].dtypes)
print("Duplicate customer_state values:", regional_performance['customer_state'].duplicated().sum())

Missing values in both rank columns: 
 regional_order_count_rank    0
regional_gmv_rank            0
dtype: int64
Minimum order-count rank: 1.0
Minimum GMV rank: 1.0
Maximum order-count rank: 27.0
Maximum GMV rank: 27.0
Data types of both rank columns: 
 regional_order_count_rank    float64
regional_gmv_rank            float64
dtype: object
Duplicate customer_state values: 0


In [24]:
# Inspect the highest- and lowest-performing states by delivered-order volume and GMV.
highest_order_volume_state = regional_performance[regional_performance['regional_order_count_rank']==regional_performance['regional_order_count_rank'].min()]
lowest_order_volume_state = regional_performance[regional_performance['regional_order_count_rank']==regional_performance['regional_order_count_rank'].max()]
highest_gmv_state = regional_performance[regional_performance['regional_gmv_rank']==regional_performance['regional_gmv_rank'].min()]
lowest_gmv_state = regional_performance[regional_performance['regional_gmv_rank']==regional_performance['regional_gmv_rank'].max()]

regional_state_comparison_columns = ['customer_state','regional_order_count', 'regional_order_share_percentage', 'regional_gmv', 'regional_gmv_share_percentage', 'regional_order_count_rank', 'regional_gmv_rank']

In [25]:
highest_order_volume_state[regional_state_comparison_columns]

,customer_state,regional_order_count,regional_order_share_percentage,regional_gmv,regional_gmv_share_percentage,regional_order_count_rank,regional_gmv_rank
25,SP,40501,41.979519,5067633.16,38.328736,1.0,1.0


In [26]:
lowest_order_volume_state[regional_state_comparison_columns]

,customer_state,regional_order_count,regional_order_share_percentage,regional_gmv,regional_gmv_share_percentage,regional_order_count_rank,regional_gmv_rank
21,RR,41,0.042497,7057.47,0.053379,27.0,27.0


In [27]:
highest_gmv_state[regional_state_comparison_columns]

,customer_state,regional_order_count,regional_order_share_percentage,regional_gmv,regional_gmv_share_percentage,regional_order_count_rank,regional_gmv_rank
25,SP,40501,41.979519,5067633.16,38.328736,1.0,1.0


In [28]:
lowest_gmv_state[regional_state_comparison_columns]

,customer_state,regional_order_count,regional_order_share_percentage,regional_gmv,regional_gmv_share_percentage,regional_order_count_rank,regional_gmv_rank
21,RR,41,0.042497,7057.47,0.053379,27.0,27.0


In [29]:
# Validate the highest- and lowest-performing state subsets.
print("Each of the four subsets contains exactly one row:","\n",
      "highest_order_volume_state:",len(highest_order_volume_state)==1,"\n",
      "lowest_order_volume_state:",len(lowest_order_volume_state)==1,"\n",
      "highest_gmv_state:",len(highest_gmv_state)==1,"\n",
      "lowest_gmv_state:",len(lowest_gmv_state)==1)
print("\n")

print("The highest-order subset has order rank 1:", highest_order_volume_state['regional_order_count_rank'].eq(1).all())
print("\n")

print("The lowest-order subset has the maximum order rank:", lowest_order_volume_state['regional_order_count_rank'].eq(regional_performance['regional_order_count_rank'].max()).all())
print("\n")

print("The highest-GMV subset has GMV rank 1:", highest_gmv_state['regional_gmv_rank'].eq(1).all())
print("\n")

print("The lowest-GMV subset has the maximum GMV rank:", lowest_gmv_state['regional_gmv_rank'].eq(regional_performance['regional_gmv_rank'].max()).all())
print("\n")

print("The highest-order and highest-GMV state names match:",set(highest_order_volume_state['customer_state'])==set(highest_gmv_state['customer_state']))
print("\n")

print("The lowest-order and lowest-GMV state names match:", set(lowest_order_volume_state['customer_state'])==set(lowest_gmv_state['customer_state']))

Each of the four subsets contains exactly one row: 
 highest_order_volume_state: True 
 lowest_order_volume_state: True 
 highest_gmv_state: True 
 lowest_gmv_state: True


The highest-order subset has order rank 1: True


The lowest-order subset has the maximum order rank: True


The highest-GMV subset has GMV rank 1: True


The lowest-GMV subset has the maximum GMV rank: True


The highest-order and highest-GMV state names match: True


The lowest-order and lowest-GMV state names match: True


In [30]:
# Examine delivered-order and GMV distributions across states.
regional_performance[['regional_order_count', 'regional_gmv']].describe()

,regional_order_count,regional_gmv
count,27.000000,2.700000e+01
mean,3573.259259,4.896851e+05
std,8021.983682,1.012662e+06
min,41.000000,7.057470e+03
25%,366.000000,6.771496e+04
50%,886.000000,1.521916e+05
75%,2668.000000,3.950413e+05
max,40501.000000,5.067633e+06


In [31]:
# Flag states at or below the delivered-order-count 25th percentile as low-volume candidates.
regional_performance['low_regional_order_volume_candidate'] = regional_performance['regional_order_count'] <= regional_performance['regional_order_count'].quantile(0.25)

In [32]:
# Validate the regional low-order-volume candidate classification.
print("The calculated 25th-percentile threshold: ", regional_performance['regional_order_count'].quantile(0.25))
print("Number of states flagged as low-volume candidates: ", regional_performance['low_regional_order_volume_candidate'].sum())
print("Missing values in low_regional_order_volume_candidate: ", regional_performance['low_regional_order_volume_candidate'].isna().sum())
print("low_regional_order_volume_candidate data type: ", regional_performance['low_regional_order_volume_candidate'].dtype)

low_regional_order_volume_candidates = regional_performance[regional_performance['low_regional_order_volume_candidate']]
print("Every flagged state has regional_order_count at or below the threshold: ", (low_regional_order_volume_candidates['regional_order_count']<= regional_performance['regional_order_count'].quantile(0.25)).all())

print("The maximum order count among flagged states: ", low_regional_order_volume_candidates['regional_order_count'].max())

The calculated 25th-percentile threshold:  366.0
Number of states flagged as low-volume candidates:  7
Missing values in low_regional_order_volume_candidate:  0
low_regional_order_volume_candidate data type:  bool
Every flagged state has regional_order_count at or below the threshold:  True
The maximum order count among flagged states:  335


In [33]:
# Flag states at or below the GMV 25th percentile as low-GMV candidates.
regional_performance['low_regional_gmv_candidate'] = regional_performance['regional_gmv'] <= regional_performance['regional_gmv'].quantile(0.25)

In [34]:
# Validate the regional low-gmv candidate classification.
print("The calculated 25th-percentile threshold: ", regional_performance['regional_gmv'].quantile(0.25))
print("Number of states flagged as low-gmv candidates: ", regional_performance['low_regional_gmv_candidate'].sum())
print("Missing values in low_regional_gmv_candidate: ", regional_performance['low_regional_gmv_candidate'].isna().sum())
print("low_regional_gmv_candidate data type: ", regional_performance['low_regional_gmv_candidate'].dtype)

low_regional_gmv_candidates = regional_performance[regional_performance['low_regional_gmv_candidate']]
print("Every flagged state has regional_gmv at or below the threshold: ", (low_regional_gmv_candidates['regional_gmv']<= regional_performance['regional_gmv'].quantile(0.25)).all())

print("The maximum gmv among flagged states: ", low_regional_gmv_candidates['regional_gmv'].max())

The calculated 25th-percentile threshold:  67714.955
Number of states flagged as low-gmv candidates:  7
Missing values in low_regional_gmv_candidate:  0
low_regional_gmv_candidate data type:  bool
Every flagged state has regional_gmv at or below the threshold:  True
The maximum gmv among flagged states:  56574.19


In [35]:
# Flag states that fall in the bottom quartile for both delivered-order volume and GMV.
regional_performance['low_regional_performance_candidate'] = (regional_performance['low_regional_order_volume_candidate'] & regional_performance['low_regional_gmv_candidate'] == True)

In [36]:
# Validate the final low-performance regional classification.
print("Number of final regional candidates: ", regional_performance['low_regional_performance_candidate'].sum())
print("Missing values: ", regional_performance['low_regional_performance_candidate'].isna().sum())
print("Data Type: ", regional_performance['low_regional_performance_candidate'].dtype)

final_low_regional_candidates = regional_performance[regional_performance['low_regional_performance_candidate']==True]
print("Every final candidate has both component flags equal to True: ", final_low_regional_candidates[['low_regional_order_volume_candidate', 'low_regional_gmv_candidate']].all().all())

print("Every final candidate is at or below both the order-count and GMV thresholds: ", ((final_low_regional_candidates['regional_order_count']<=regional_performance['regional_order_count'].quantile(0.25))&(final_low_regional_candidates['regional_gmv']<=regional_performance['regional_gmv'].quantile(0.25))).all())

Number of final regional candidates:  7
Missing values:  0
Data Type:  bool
Every final candidate has both component flags equal to True:  True
Every final candidate is at or below both the order-count and GMV thresholds:  True


In [37]:
# Create the final state-level low-performance candidate table.
low_performance_regional_candidates = regional_performance[regional_performance['low_regional_performance_candidate']==True].copy().sort_values(by=['regional_order_count', 'regional_gmv'], ascending=True).reset_index(drop=True)
low_performance_regional_candidates

,customer_state,regional_item_count,regional_gmv,regional_freight_value,regional_order_count,regional_customer_count,regional_average_delivery_duration_days,regional_valid_delivery_order_count,regional_late_order_count,regional_reviewed_order_count,...,regional_gmv_share_percentage,regional_freight_burden_percentage,regional_late_delivery_rate,regional_review_coverage_rate,regional_review_response_over_30_days_rate,regional_order_count_rank,regional_gmv_rank,low_regional_order_volume_candidate,low_regional_gmv_candidate,low_regional_performance_candidate
0,RR,46,7057.47,1982.05,41,40,29.387546,41,5,41,...,0.053379,28.084427,12.195122,100.000000,0.0,27.0,27.0,True,True,True
1,AP,81,13374.81,2767.00,67,66,27.185068,67,2,66,...,0.101160,20.688144,2.985075,98.507463,1.515152,26.0,26.0,True,True,True
2,AC,91,15930.97,3644.36,80,76,21.035713,80,3,80,...,0.120493,22.875945,3.750000,100.000000,0.0,25.0,25.0,True,True,True
3,AM,163,22155.84,5429.63,145,140,26.425991,145,4,144,...,0.167574,24.506541,2.758621,99.310345,0.694444,24.0,24.0,True,True,True
4,RO,273,45682.76,11283.24,243,231,19.372037,243,7,242,...,0.345519,24.699121,2.880658,99.588477,0.826446,23.0,23.0,True,True,True
5,TO,310,48402.51,11604.86,274,267,17.658063,274,27,273,...,0.366089,23.975740,9.854015,99.635036,1.098901,22.0,22.0,True,True,True
6,SE,375,56574.19,13714.94,335,329,21.519788,335,51,334,...,0.427895,24.242397,15.223881,99.701493,0.898204,21.0,21.0,True,True,True


In [38]:
regional_candidate_columns = [
    'customer_state',
    'regional_order_count',
    'regional_order_share_percentage',
    'regional_customer_count',
    'regional_item_count',
    'regional_gmv',
    'regional_gmv_share_percentage',
    'regional_average_order_value',
    'regional_freight_burden_percentage',
    'regional_average_delivery_duration_days',
    'regional_late_delivery_rate',
    'regional_average_review_score',
    'regional_review_coverage_rate',
    'regional_review_response_over_30_days_rate'
]
low_performance_regional_candidates[regional_candidate_columns]

,customer_state,regional_order_count,regional_order_share_percentage,regional_customer_count,regional_item_count,regional_gmv,regional_gmv_share_percentage,regional_average_order_value,regional_freight_burden_percentage,regional_average_delivery_duration_days,regional_late_delivery_rate,regional_average_review_score,regional_review_coverage_rate,regional_review_response_over_30_days_rate
0,RR,41,0.042497,40,46,7057.47,0.053379,172.133415,28.084427,29.387546,12.195122,3.902439,100.000000,0.0
1,AP,67,0.069446,66,81,13374.81,0.101160,199.624030,20.688144,27.185068,2.985075,4.242424,98.507463,1.515152
2,AC,80,0.082920,76,91,15930.97,0.120493,199.137125,22.875945,21.035713,3.750000,4.087500,100.000000,0.0
3,AM,145,0.150293,140,163,22155.84,0.167574,152.798897,24.506541,26.425991,2.758621,4.243056,99.310345,0.694444
4,RO,243,0.251871,231,273,45682.76,0.345519,187.994897,24.699121,19.372037,2.880658,4.165289,99.588477,0.826446
5,TO,274,0.284003,267,310,48402.51,0.366089,176.651496,23.975740,17.658063,9.854015,4.153846,99.635036,1.098901
6,SE,335,0.347229,329,375,56574.19,0.427895,168.878179,24.242397,21.519788,15.223881,3.907186,99.701493,0.898204


In [39]:
# Validate the final state-level low-performance candidate table.
print("Table shape:",low_performance_regional_candidates.shape)

print("Duplicate customer_state count:", low_performance_regional_candidates['customer_state'].duplicated().sum())

print("Missing values across the table:", low_performance_regional_candidates.isna().sum().sum())

print("Every low_regional_performance_candidate value is True:", low_performance_regional_candidates['low_regional_performance_candidate'].all())

print("Every low_regional_order_volume_candidate value is True:", low_performance_regional_candidates['low_regional_order_volume_candidate'].all())

print("Every low_regional_gmv_candidate value is True:", low_performance_regional_candidates['low_regional_gmv_candidate'].all())

regional_order_count_q1 = regional_performance['regional_order_count'].quantile(0.25)

regional_gmv_q1 = regional_performance['regional_gmv'].quantile(0.25)

print("Every row satisfies both quartile thresholds:", ((low_performance_regional_candidates['regional_order_count'] <= regional_order_count_q1) & (low_performance_regional_candidates['regional_gmv']<= regional_gmv_q1)).all())

print("Number of represented states:", low_performance_regional_candidates['customer_state'].nunique())

print("Order counts are sorted in ascending order:", low_performance_regional_candidates['regional_order_count'].is_monotonic_increasing)

Table shape: (7, 26)
Duplicate customer_state count: 0
Missing values across the table: 0
Every low_regional_performance_candidate value is True: True
Every low_regional_order_volume_candidate value is True: True
Every low_regional_gmv_candidate value is True: True
Every row satisfies both quartile thresholds: True
Number of represented states: 7
Order counts are sorted in ascending order: True


In [40]:
# Inspect commercial, delivery, freight, and review measures for the final regional candidates.
low_performance_regional_candidates[regional_candidate_columns]

,customer_state,regional_order_count,regional_order_share_percentage,regional_customer_count,regional_item_count,regional_gmv,regional_gmv_share_percentage,regional_average_order_value,regional_freight_burden_percentage,regional_average_delivery_duration_days,regional_late_delivery_rate,regional_average_review_score,regional_review_coverage_rate,regional_review_response_over_30_days_rate
0,RR,41,0.042497,40,46,7057.47,0.053379,172.133415,28.084427,29.387546,12.195122,3.902439,100.000000,0.0
1,AP,67,0.069446,66,81,13374.81,0.101160,199.624030,20.688144,27.185068,2.985075,4.242424,98.507463,1.515152
2,AC,80,0.082920,76,91,15930.97,0.120493,199.137125,22.875945,21.035713,3.750000,4.087500,100.000000,0.0
3,AM,145,0.150293,140,163,22155.84,0.167574,152.798897,24.506541,26.425991,2.758621,4.243056,99.310345,0.694444
4,RO,243,0.251871,231,273,45682.76,0.345519,187.994897,24.699121,19.372037,2.880658,4.165289,99.588477,0.826446
5,TO,274,0.284003,267,310,48402.51,0.366089,176.651496,23.975740,17.658063,9.854015,4.153846,99.635036,1.098901
6,SE,335,0.347229,329,375,56574.19,0.427895,168.878179,24.242397,21.519788,15.223881,3.907186,99.701493,0.898204


### Regional Low-Performance Screening Summary

Seven states—RR, AP, AC, AM, RO, TO, and SE—fall at or below the state-level 25th percentiles for both delivered-order volume and GMV. These states are therefore classified as low-performance candidates based on commercial scale.

Operational measures show that the candidates do not share the same customer-experience profile. RR combines the lowest commercial contribution with the highest freight burden and longest average delivery duration among the candidates. SE records the highest candidate late-delivery rate and a comparatively low average review score, while TO also shows an elevated late-delivery rate. In contrast, AP and AM maintain relatively strong review scores despite longer delivery durations.

This classification is a screening result rather than definitive evidence of poor regional performance. Population size, market penetration, logistics infrastructure, profitability, and strategic importance are unavailable in the dataset and may explain part of the observed differences.

## 4. Create the Customer-Performance Table

Customer performance is evaluated at the `customer_unique_id` level so that purchases made by the same customer across multiple orders are combined. Item-level records are used for delivered-item count, GMV, and freight value, while the one-row-per-order scope is used for order frequency, purchase dates, delivery performance, and review measures.

Because a small number of customers placed orders from more than one state, each customer is assigned the state associated with their most recent delivered order for customer-level reporting. This preserves one row per customer without duplicating customer metrics across states.

In [41]:
# Validate the purchase-date field required for customer recency and state attribution.
print("Data type: ", regional_order_scope['order_purchase_timestamp'].dtype)
print("Missing value count: ", regional_order_scope['order_purchase_timestamp'].isna().sum())
print("Minimum purchase timestamp: ", regional_order_scope['order_purchase_timestamp'].min())
print("Maximum purchase timestamp: ", regional_order_scope['order_purchase_timestamp'].max())
print("Whether the maximum date is greater than or equal to the minimum date: ", regional_order_scope['order_purchase_timestamp'].max() >= regional_order_scope['order_purchase_timestamp'].min())

Data type:  datetime64[us]
Missing value count:  0
Minimum purchase timestamp:  2016-09-15 12:16:38
Maximum purchase timestamp:  2018-08-29 15:00:37
Whether the maximum date is greater than or equal to the minimum date:  True


### 4.1 Aggregate Customer Item-Level Measures

Delivered-item records are grouped by `customer_unique_id` to calculate each customer’s purchased-item volume, product and category variety, merchandise value, and freight value. These measures remain at item grain so that every delivered item and its associated commercial values are included.

In [42]:
# Aggregate delivered-item and commercial measures by unique customer.
customer_item_performance = delivered_order_items.groupby('customer_unique_id').agg(customer_item_count=('order_item_id', 'count'),
                                                                                    customer_product_count=('product_id', 'nunique'),
                                                                                    customer_available_category_count=('product_category_name_english', 'nunique'),
                                                                                    customer_gmv=('price', 'sum'),
                                                                                    customer_freight_value=('freight_value', 'sum')
                                                                                    ).reset_index()

In [43]:
customer_item_performance

,customer_unique_id,customer_item_count,customer_product_count,customer_available_category_count,customer_gmv,customer_freight_value
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,1,129.90,12.00
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,1,18.90,8.29
2,0000f46a3911fa3c0805444483337064,1,1,1,69.00,17.22
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,1,25.99,17.63
4,0004aac84e0df4da2b147fca70cf8255,1,1,1,180.00,16.89
...,...,...,...,...,...,...
93353,fffcf5a5ff07b0908bd4e2dbc735a684,2,2,1,1570.00,497.42
93354,fffea47cd6d3cc0a88bd621562a9d061,1,1,1,64.89,19.69
93355,ffff371b4d645b6ecea244b27531430a,1,1,1,89.90,22.56
93356,ffff5962728ec6157033ef9805bacc48,1,1,1,115.00,18.69


In [44]:
# Validate the customer item-level performance table.
print('Table Shape: ', customer_item_performance.shape)
print('Duplicate customer_unique_id values: ', customer_item_performance['customer_unique_id'].duplicated().sum())
print('Missing values across the table: ', customer_item_performance.isna().sum().sum())
print('Sum of customer_item_count: ', customer_item_performance['customer_item_count'].sum())
print('Sum of customer_gmv same as sum of price in delivered_order_items: ', np.isclose(customer_item_performance['customer_gmv'].sum(), delivered_order_items['price'].sum()))
print('Sum of customer_freight_value same as sum of freight in delivered_order_items: ', np.isclose(customer_item_performance['customer_freight_value'].sum(), delivered_order_items['freight_value'].sum()))
print('Minimum customer_item_count: ', customer_item_performance['customer_item_count'].min())
print('Minimum customer_product_count: ', customer_item_performance['customer_product_count'].min())
print('Minimum customer_available_category_count: ', customer_item_performance['customer_available_category_count'].min())


Table Shape:  (93358, 6)
Duplicate customer_unique_id values:  0
Missing values across the table:  0
Sum of customer_item_count:  110197
Sum of customer_gmv same as sum of price in delivered_order_items:  True
Sum of customer_freight_value same as sum of freight in delivered_order_items:  True
Minimum customer_item_count:  1
Minimum customer_product_count:  1
Minimum customer_available_category_count:  0


In [45]:
# Investigate customers with no available product-category labels.
print("Number of missing product_category_name_english values in delivered_order_items: ", delivered_order_items['product_category_name_english'].isna().sum())
print("Number of customers with customer_available_category_count == 0: ", (customer_item_performance['customer_available_category_count']==0).sum())

Number of missing product_category_name_english values in delivered_order_items:  1559
Number of customers with customer_available_category_count == 0:  1279


In [46]:
# Create a customer-level subset where no available category was counted.
customers_without_available_category = customer_item_performance[customer_item_performance['customer_available_category_count'] == 0].copy()

print("Number of customers without an available category:", len(customers_without_available_category))

# Confirm that every customer in the subset has at least one delivered item.
print("Every customer has at least one delivered item:", customers_without_available_category['customer_item_count'].ge(1).all())

# Retrieve the corresponding item-level records.
customers_without_available_category_items = delivered_order_items[delivered_order_items['customer_unique_id'].isin(customers_without_available_category['customer_unique_id'])].copy()

# Display the first few underlying item records.
customers_without_available_category_items[['customer_unique_id', 'order_id', 'product_id', 'product_category_name_english']].head()

Number of customers without an available category: 1279
Every customer has at least one delivered item: True


,customer_unique_id,order_id,product_id,product_category_name_english
121,b0efd7adeb04d15b16edaedd11215d40,0046e1d57f4c07c8c92ab26be8c3dfc0,ff6caf9340512b8bf6d2a2a6df032cfa,NaN
123,d4c784ecad19dfe09e63ffaf9ada82a3,00482f2670787292280e0a8153d82467,a9c404971d1a5b1cbc2e4070e02731fd,NaN
130,17726716a98c98a7806b43a789820245,004f5d8f238e8908e6864b874eda3391,5a848e4ab52fd5445cdc07aab1c40e48,NaN
140,9a5b186787a143669398bd8083720c21,0057199db02d1a5ef41bacbf41f8f63b,41eee23c25f7a574dfaf8d5c151dbb12,NaN
169,d5faae3f913e775994e53068992969b5,006cb7cafc99b29548d4f412c7f9f493,e10758160da97891c2fdcbc35f0f031d,NaN


In [47]:
print("Every category label is missing:", customers_without_available_category_items['product_category_name_english'].isna().all())

Every category label is missing: True


### 4.2 Aggregate Customer Order-Level Measures

The one-row-per-delivered-order scope is grouped by `customer_unique_id` to calculate customer purchase frequency, first and most recent purchase dates, delivery performance, and review outcomes. Using the order-level scope prevents order-level dates, delivery durations, and review measures from being duplicated across multi-item orders.

In [48]:
# Aggregate delivered-order, purchase-date, delivery, and review measures by unique customer.
customer_order_performance = regional_order_scope.groupby('customer_unique_id').agg(customer_order_count = ('order_id', 'count'),
                                                                                    customer_first_purchase_timestamp= ('order_purchase_timestamp', 'min'),
                                                                                    customer_last_purchase_timestamp= ('order_purchase_timestamp', 'max'),
                                                                                    customer_valid_delivery_order_count= ('valid_delivery_order', 'sum'),
                                                                                    customer_late_order_count= ('late_delivery_order', 'sum'),
                                                                                    customer_average_delivery_duration_days= ('delivery_duration_days', 'mean'),
                                                                                    customer_reviewed_order_count= ('reviewed_order', 'sum'),
                                                                                    customer_average_review_score= ('average_review_score', 'mean'),
                                                                                    customer_review_response_over_30_days_order_count=('flag_review_response_over_30_days', 'sum')
                                                                                    ).reset_index()

In [49]:
customer_order_performance.head()

,customer_unique_id,customer_order_count,customer_first_purchase_timestamp,customer_last_purchase_timestamp,customer_valid_delivery_order_count,customer_late_order_count,customer_average_delivery_duration_days,customer_reviewed_order_count,customer_average_review_score,customer_review_response_over_30_days_order_count
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,2018-05-10 10:56:27,1,0,6.411227,1,5.0,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,2018-05-07 11:11:27,1,0,3.285590,1,4.0,0
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,2017-03-10 21:05:03,1,0,25.731759,1,3.0,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,2017-10-12 20:29:41,1,0,20.037083,1,4.0,0
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,2017-11-14 19:45:42,1,0,13.141134,1,5.0,0


In [50]:
# Validate the customer order-level performance table.
print("Table shape:", customer_order_performance.shape)

print("Duplicate customer_unique_id values:", customer_order_performance['customer_unique_id'].duplicated().sum())

print("Sum of customer_order_count:", customer_order_performance['customer_order_count'].sum())

print("Sum of customer_valid_delivery_order_count:", customer_order_performance['customer_valid_delivery_order_count'].sum())

print("Sum of customer_late_order_count:", customer_order_performance['customer_late_order_count'].sum())

print("Sum of customer_reviewed_order_count:", customer_order_performance['customer_reviewed_order_count'].sum())

print("Sum of customer_review_response_over_30_days_order_count:", customer_order_performance['customer_review_response_over_30_days_order_count'].sum())

print("\nMissing-value counts:", "\n",customer_order_performance.isna().sum())

print("\nEvery last-purchase timestamp is greater than or equal to " "the first-purchase timestamp:", (customer_order_performance['customer_last_purchase_timestamp'] >= customer_order_performance['customer_first_purchase_timestamp']).all())

print("\nMinimum customer_average_review_score:",customer_order_performance['customer_average_review_score'].min())

print("Maximum customer_average_review_score:", customer_order_performance['customer_average_review_score'].max())

print("All available average review scores are between 1 and 5:", customer_order_performance['customer_average_review_score'].dropna().between(1, 5).all())

Table shape: (93358, 10)
Duplicate customer_unique_id values: 0
Sum of customer_order_count: 96478
Sum of customer_valid_delivery_order_count: 96470
Sum of customer_late_order_count: 6534
Sum of customer_reviewed_order_count: 95832
Sum of customer_review_response_over_30_days_order_count: 645

Missing-value counts: 
 customer_unique_id                                     0
customer_order_count                                   0
customer_first_purchase_timestamp                      0
customer_last_purchase_timestamp                       0
customer_valid_delivery_order_count                    0
customer_late_order_count                              0
customer_average_delivery_duration_days                8
customer_reviewed_order_count                          0
customer_average_review_score                        603
customer_review_response_over_30_days_order_count      0
dtype: int64

Every last-purchase timestamp is greater than or equal to the first-purchase timestamp: True

Min

In [51]:
# Confirm that missing customer averages correspond to zero valid observations.

customers_with_missing_delivery_average = customer_order_performance[customer_order_performance['customer_average_delivery_duration_days'].isna()]
print("Missing delivery averages have zero valid delivery orders:", customers_with_missing_delivery_average['customer_valid_delivery_order_count'].eq(0).all())

customers_with_zero_valid_delivery_orders = customer_order_performance[customer_order_performance['customer_valid_delivery_order_count'].eq(0)]
print("Zero valid delivery orders have missing delivery averages:", customers_with_zero_valid_delivery_orders['customer_average_delivery_duration_days'].isna().all())

customers_with_missing_review_average = customer_order_performance[customer_order_performance['customer_average_review_score'].isna()]
print("Missing review averages have zero reviewed orders:", customers_with_missing_review_average['customer_reviewed_order_count'].eq(0).all())

customers_with_zero_reviewed_orders = customer_order_performance[customer_order_performance['customer_reviewed_order_count'].eq(0)]
print("Zero reviewed orders have missing review averages:", customers_with_zero_reviewed_orders['customer_average_review_score'].isna().all())

Missing delivery averages have zero valid delivery orders: True
Zero valid delivery orders have missing delivery averages: True
Missing review averages have zero reviewed orders: True
Zero reviewed orders have missing review averages: True


### 4.3 Assign Each Customer’s Most Recent State

Most customers are associated with only one state, but 37 `customer_unique_id` values appear in more than one state across their delivered orders. To preserve one row per customer, each customer is assigned the state from their most recent delivered order. Purchase timestamp is used to establish recency, with `order_id` used as a deterministic tie-breaker when timestamps are equal.

In [52]:
# Assign each customer the state from their most recent delivered order.
customer_latest_state = (regional_order_scope.sort_values(by=['customer_unique_id','order_purchase_timestamp','order_id'],ascending=True)
                         .drop_duplicates(subset='customer_unique_id',keep='last')[['customer_unique_id','customer_state','order_purchase_timestamp']]
                         .rename(columns={'customer_state': 'customer_latest_state','order_purchase_timestamp':'customer_latest_state_purchase_timestamp'})
                         .reset_index(drop=True))

customer_latest_state.head()

,customer_unique_id,customer_latest_state,customer_latest_state_purchase_timestamp
0,0000366f3b9a7992bf8c76cfdf3221e2,SP,2018-05-10 10:56:27
1,0000b849f77a49e4a4ce2b2a4ca5be3f,SP,2018-05-07 11:11:27
2,0000f46a3911fa3c0805444483337064,SC,2017-03-10 21:05:03
3,0000f6ccb0745a6a4b88665a16c9f078,PA,2017-10-12 20:29:41
4,0004aac84e0df4da2b147fca70cf8255,SP,2017-11-14 19:45:42


In [53]:
# Validate the most-recent customer-state attribution table.
print("Table shape:", customer_latest_state.shape)

print("Duplicate customer_unique_id values:", customer_latest_state['customer_unique_id'].duplicated().sum())

print("Missing values across the table:", customer_latest_state.isna().sum().sum())

print("Distinct latest states:", customer_latest_state['customer_latest_state'].nunique())

customer_latest_state_validation = customer_latest_state.merge(customer_order_performance[['customer_unique_id', 'customer_last_purchase_timestamp']],on='customer_unique_id',how='inner',validate='one_to_one')

print("Validation-table shape:", customer_latest_state_validation.shape)

print("Every selected state timestamp matches the customer's last-purchase timestamp:", (customer_latest_state_validation['customer_latest_state_purchase_timestamp'] == customer_latest_state_validation['customer_last_purchase_timestamp']).all())


Table shape: (93358, 3)
Duplicate customer_unique_id values: 0
Missing values across the table: 0
Distinct latest states: 27
Validation-table shape: (93358, 4)
Every selected state timestamp matches the customer's last-purchase timestamp: True


### 4.4 Combine Customer-Level Performance Measures

Customer item-level measures, order-level measures, and most-recent state attribution are combined using `customer_unique_id`. Each source table contains one row per unique customer, allowing one-to-one merges without changing the customer-level analytical grain.

In [54]:
# Combine customer item-level, order-level, and latest-state measures.
customer_performance = customer_item_performance.merge(customer_order_performance, on='customer_unique_id', how='inner', validate='one_to_one')

customer_performance = customer_performance.merge(customer_latest_state[['customer_unique_id', 'customer_latest_state']], on='customer_unique_id', how='inner', validate='one_to_one')

customer_performance.head()

,customer_unique_id,customer_item_count,customer_product_count,customer_available_category_count,customer_gmv,customer_freight_value,customer_order_count,customer_first_purchase_timestamp,customer_last_purchase_timestamp,customer_valid_delivery_order_count,customer_late_order_count,customer_average_delivery_duration_days,customer_reviewed_order_count,customer_average_review_score,customer_review_response_over_30_days_order_count,customer_latest_state
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,1,129.90,12.00,1,2018-05-10 10:56:27,2018-05-10 10:56:27,1,0,6.411227,1,5.0,0,SP
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,1,18.90,8.29,1,2018-05-07 11:11:27,2018-05-07 11:11:27,1,0,3.285590,1,4.0,0,SP
2,0000f46a3911fa3c0805444483337064,1,1,1,69.00,17.22,1,2017-03-10 21:05:03,2017-03-10 21:05:03,1,0,25.731759,1,3.0,0,SC
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,1,25.99,17.63,1,2017-10-12 20:29:41,2017-10-12 20:29:41,1,0,20.037083,1,4.0,0,PA
4,0004aac84e0df4da2b147fca70cf8255,1,1,1,180.00,16.89,1,2017-11-14 19:45:42,2017-11-14 19:45:42,1,0,13.141134,1,5.0,0,SP


In [55]:
# Validate the combined customer-performance table before deriving metrics.
print("Table shape:", customer_performance.shape)

print("Duplicate customer_unique_id values:", customer_performance['customer_unique_id'].duplicated().sum())

print("Distinct customer_unique_id values:", customer_performance['customer_unique_id'].nunique())

print("Distinct latest states:", customer_performance['customer_latest_state'].nunique())

print("\nMissing-value counts:","\n",customer_performance.isna().sum())

print("\nSum of customer_item_count:",customer_performance['customer_item_count'].sum())


Table shape: (93358, 16)
Duplicate customer_unique_id values: 0
Distinct customer_unique_id values: 93358
Distinct latest states: 27

Missing-value counts: 
 customer_unique_id                                     0
customer_item_count                                    0
customer_product_count                                 0
customer_available_category_count                      0
customer_gmv                                           0
customer_freight_value                                 0
customer_order_count                                   0
customer_first_purchase_timestamp                      0
customer_last_purchase_timestamp                       0
customer_valid_delivery_order_count                    0
customer_late_order_count                              0
customer_average_delivery_duration_days                8
customer_reviewed_order_count                          0
customer_average_review_score                        603
customer_review_response_over_30_days_order_

In [56]:
print("Sum of customer_order_count:",customer_performance['customer_order_count'].sum())

print("Customer GMV reconciles with delivered-item GMV:", np.isclose(customer_performance['customer_gmv'].sum(), delivered_order_items['price'].sum()))

print("Customer freight value reconciles with delivered-item freight:", np.isclose(customer_performance['customer_freight_value'].sum(), delivered_order_items['freight_value'].sum()))

print("Sum of customer_late_order_count:", customer_performance['customer_late_order_count'].sum())

print("Sum of customer_reviewed_order_count:", customer_performance['customer_reviewed_order_count'].sum())

Sum of customer_order_count: 96478
Customer GMV reconciles with delivered-item GMV: True
Customer freight value reconciles with delivered-item freight: True
Sum of customer_late_order_count: 6534
Sum of customer_reviewed_order_count: 95832


### 4.5 Calculate Derived Customer-Performance Metrics

Customer-level commercial, engagement, delivery, and review measures are derived from the validated additive totals. Average order value and items per order describe purchasing behavior, while freight burden, late-delivery rate, review coverage, and delayed review-response rate describe customer experience. Purchase timestamps are also used to measure customer relationship duration and recency relative to the final purchase timestamp in the dataset.

In [57]:
# Calculate customer commercial, delivery, review, relationship, and recency metrics.

analysis_reference_timestamp = regional_order_scope['order_purchase_timestamp'].max()

customer_performance['customer_average_order_value'] = (customer_performance['customer_gmv'] / customer_performance['customer_order_count'])

customer_performance['customer_average_items_per_order'] = (customer_performance['customer_item_count'] / customer_performance['customer_order_count'])

customer_performance['customer_freight_burden_percentage'] = (customer_performance['customer_freight_value'] / customer_performance['customer_gmv']) * 100

customer_performance['customer_late_delivery_rate'] = (customer_performance['customer_late_order_count'] / customer_performance['customer_valid_delivery_order_count'].where(customer_performance['customer_valid_delivery_order_count'].gt(0))) * 100

customer_performance['customer_review_coverage_rate'] = (customer_performance['customer_reviewed_order_count'] / customer_performance['customer_order_count']) * 100

customer_performance['customer_review_response_over_30_days_rate'] = (customer_performance['customer_review_response_over_30_days_order_count'] / customer_performance['customer_reviewed_order_count'].where(customer_performance['customer_reviewed_order_count'].gt(0))) * 100

customer_performance['customer_relationship_duration_days'] = ((customer_performance['customer_last_purchase_timestamp']- customer_performance['customer_first_purchase_timestamp']).dt.total_seconds()/ 86400)

customer_performance['customer_recency_days'] = ((analysis_reference_timestamp - customer_performance['customer_last_purchase_timestamp']).dt.total_seconds()/ 86400)

customer_performance.head()

,customer_unique_id,customer_item_count,customer_product_count,customer_available_category_count,customer_gmv,customer_freight_value,customer_order_count,customer_first_purchase_timestamp,customer_last_purchase_timestamp,customer_valid_delivery_order_count,...,customer_review_response_over_30_days_order_count,customer_latest_state,customer_average_order_value,customer_average_items_per_order,customer_freight_burden_percentage,customer_late_delivery_rate,customer_review_coverage_rate,customer_review_response_over_30_days_rate,customer_relationship_duration_days,customer_recency_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,1,1,129.90,12.00,1,2018-05-10 10:56:27,2018-05-10 10:56:27,1,...,0,SP,129.90,1.0,9.237875,0.0,100.0,0.0,0.0,111.169560
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,1,1,18.90,8.29,1,2018-05-07 11:11:27,2018-05-07 11:11:27,1,...,0,SP,18.90,1.0,43.862434,0.0,100.0,0.0,0.0,114.159144
2,0000f46a3911fa3c0805444483337064,1,1,1,69.00,17.22,1,2017-03-10 21:05:03,2017-03-10 21:05:03,1,...,0,SC,69.00,1.0,24.956522,0.0,100.0,0.0,0.0,536.746921
3,0000f6ccb0745a6a4b88665a16c9f078,1,1,1,25.99,17.63,1,2017-10-12 20:29:41,2017-10-12 20:29:41,1,...,0,PA,25.99,1.0,67.833782,0.0,100.0,0.0,0.0,320.771481
4,0004aac84e0df4da2b147fca70cf8255,1,1,1,180.00,16.89,1,2017-11-14 19:45:42,2017-11-14 19:45:42,1,...,0,SP,180.00,1.0,9.383333,0.0,100.0,0.0,0.0,287.802025


In [58]:
# Validate the derived customer-performance metrics.
customer_derived_metric_columns = [
    'customer_average_order_value',
    'customer_average_items_per_order',
    'customer_freight_burden_percentage',
    'customer_late_delivery_rate',
    'customer_review_coverage_rate',
    'customer_review_response_over_30_days_rate',
    'customer_relationship_duration_days',
    'customer_recency_days'
]

print("Derived metric datatypes:", "\n", customer_performance[customer_derived_metric_columns].dtypes)

print("\nMissing-value counts:", "\n", customer_performance[customer_derived_metric_columns].isna().sum())

print("\nInfinite-value count:",np.isinf(customer_performance[customer_derived_metric_columns].astype('float64')).sum().sum())

print("\nEvery customer_average_order_value is greater than 0:", customer_performance['customer_average_order_value'].gt(0).all())

Derived metric datatypes: 
 customer_average_order_value                  float64
customer_average_items_per_order              float64
customer_freight_burden_percentage            float64
customer_late_delivery_rate                   float64
customer_review_coverage_rate                 float64
customer_review_response_over_30_days_rate    Float64
customer_relationship_duration_days           float64
customer_recency_days                         float64
dtype: object

Missing-value counts: 
 customer_average_order_value                    0
customer_average_items_per_order                0
customer_freight_burden_percentage              0
customer_late_delivery_rate                     8
customer_review_coverage_rate                   0
customer_review_response_over_30_days_rate    603
customer_relationship_duration_days             0
customer_recency_days                           0
dtype: int64

Infinite-value count: 0

Every customer_average_order_value is greater than 0: True


In [59]:
print("Every customer_average_items_per_order is at least 1:", customer_performance['customer_average_items_per_order'].ge(1).all())

print("Every customer_freight_burden_percentage is non-negative:", customer_performance['customer_freight_burden_percentage'].ge(0).all())

print("All available late-delivery rates are between 0 and 100:", customer_performance['customer_late_delivery_rate'].dropna().between(0, 100).all())

print("All review-coverage rates are between 0 and 100:", customer_performance['customer_review_coverage_rate'].between(0, 100).all())

print("All available response-over-30-days rates are between 0 and 100:", customer_performance['customer_review_response_over_30_days_rate'].dropna().between(0, 100).all())

print("All customer relationship durations are non-negative:", customer_performance['customer_relationship_duration_days'].ge(0).all())

print("All customer recency values are non-negative:", customer_performance['customer_recency_days'].ge(0).all())

print("Minimum customer recency:", customer_performance['customer_recency_days'].min())

Every customer_average_items_per_order is at least 1: True
Every customer_freight_burden_percentage is non-negative: True
All available late-delivery rates are between 0 and 100: True
All review-coverage rates are between 0 and 100: True
All available response-over-30-days rates are between 0 and 100: True
All customer relationship durations are non-negative: True
All customer recency values are non-negative: True
Minimum customer recency: 0.0


### Customer-Performance Table Validation Summary

The customer-performance table contains one row for each of the 93,358 unique delivered-order customers. Customer item counts, order counts, GMV, freight value, late orders, and reviewed orders reconcile fully with the analysis-ready delivered-order dataset. Each customer is assigned the state from their most recent delivered order.

Derived commercial, delivery, review, relationship-duration, and recency measures contain no infinite values and remain within their expected ranges. Missing late-delivery and delayed-review-response rates are limited to customers with zero valid observations for the corresponding denominator. The table is therefore suitable for comparing one-time and repeat customer behavior.

## 5. Compare One-Time and Repeat Customers

Customers are classified using their number of delivered orders. A one-time customer has exactly one delivered order, while a repeat customer has more than one delivered order. The two groups are compared using customer count, orders, items, GMV, average order value, freight burden, delivery performance, review outcomes, relationship duration, and recency.

### 5.1 Classify One-Time and Repeat Customers

Customers are classified according to their delivered-order frequency. Customers with exactly one delivered order are labelled `one_time`, while customers with more than one delivered order are labelled `repeat`. This produces mutually exclusive groups that together cover the complete customer population.

In [60]:
# Examine the delivered-order-count distribution before classifying customers.
print("Descriptive statistics for customer_order_count: ", customer_performance['customer_order_count'].describe())
print("\nNumber of customers with exactly one delivered order: ", (customer_performance['customer_order_count']==1).sum())
print("Number of customers with more than one delivered order: ", (customer_performance['customer_order_count']>1).sum())
print("Maximum delivered-order count for a single customer: ", customer_performance['customer_order_count'].max())

one_time_customer_mask = (customer_performance['customer_order_count']==1)
repeat_customer_mask = (customer_performance['customer_order_count']>1)

print("Number of one-time customers:", one_time_customer_mask.sum())
print("Number of repeat customers:", repeat_customer_mask.sum())
print("Confirm that one-time plus repeat customer counts equal the total number of customers: ", one_time_customer_mask.sum() + repeat_customer_mask.sum() == len(customer_performance))
print("Every customer belongs to exactly one group:", (one_time_customer_mask ^ repeat_customer_mask).all())

Descriptive statistics for customer_order_count:  count    93358.000000
mean         1.033420
std          0.209097
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         15.000000
Name: customer_order_count, dtype: float64

Number of customers with exactly one delivered order:  90557
Number of customers with more than one delivered order:  2801
Maximum delivered-order count for a single customer:  15
Number of one-time customers: 90557
Number of repeat customers: 2801
Confirm that one-time plus repeat customer counts equal the total number of customers:  True
Every customer belongs to exactly one group: True


In [61]:
# Classify customers as one-time or repeat based on delivered-order count.
customer_performance['customer_type'] = np.where(customer_performance['customer_order_count'] == 1, 'one_time', 'repeat')

In [62]:
# validate customer_type 
print("Unique customer_type values: ", customer_performance['customer_type'].unique().tolist())
print("Missing values: ", customer_performance['customer_type'].isna().sum())
print("One-time count: ", (customer_performance['customer_type']=='one_time').sum())
print("Repeat count: ", (customer_performance['customer_type']=='repeat').sum())

one_time_orderers = customer_performance[customer_performance['customer_type']=='one_time']
print("Every one_time customer has exactly one order:", (one_time_orderers['customer_order_count'] == 1).all())

repeat_orderers = customer_performance[customer_performance['customer_type']=='repeat']
print("Every repeat customer has exactly one order:", (repeat_orderers['customer_order_count'] > 1).all())

print("Datatype of customer_type: ", customer_performance['customer_type'].dtype)

Unique customer_type values:  ['one_time', 'repeat']
Missing values:  0
One-time count:  90557
Repeat count:  2801
Every one_time customer has exactly one order: True
Every repeat customer has exactly one order: True
Datatype of customer_type:  str


### 5.2 Create the Customer-Type Comparison Table

One-time and repeat customer groups are compared using additive commercial and customer-experience measures. Group-level rates and averages are calculated from their valid aggregate numerators and denominators so that customers with different numbers of orders are weighted appropriately.

In [63]:
# Create weighted delivery-duration and review-score totals for customer-type aggregation.
customer_performance['customer_delivery_duration_total_days'] = (customer_performance['customer_average_delivery_duration_days'] * customer_performance['customer_valid_delivery_order_count']).fillna(0)

customer_performance['customer_review_score_total'] = (customer_performance['customer_average_review_score'] * customer_performance['customer_reviewed_order_count']).fillna(0)

In [64]:
# Validate the weighted customer-level helper totals.
print("Missing delivery-duration totals:", customer_performance['customer_delivery_duration_total_days'].isna().sum())

print("Missing review-score totals:", customer_performance['customer_review_score_total'].isna().sum())

print("Delivery-duration total reconciles with the order-level scope:", np.isclose(customer_performance['customer_delivery_duration_total_days'].sum(), regional_order_scope['delivery_duration_days'].sum()))

print("Review-score total reconciles with the order-level scope:", np.isclose(customer_performance['customer_review_score_total'].sum(), regional_order_scope['average_review_score'].sum()))

Missing delivery-duration totals: 0
Missing review-score totals: 0
Delivery-duration total reconciles with the order-level scope: True
Review-score total reconciles with the order-level scope: True


In [65]:
# Aggregate additive and customer-level measures by customer type.
customer_type_comparison = customer_performance.groupby('customer_type').agg(customer_count=('customer_unique_id', 'count'),
                                                                             total_order_count=('customer_order_count', 'sum'),
                                                                             total_item_count=('customer_item_count', 'sum'),
                                                                             total_gmv=('customer_gmv', 'sum'),
                                                                             total_freight_value=('customer_freight_value', 'sum'),
                                                                             valid_delivery_order_count=('customer_valid_delivery_order_count', 'sum'),
                                                                             late_order_count=('customer_late_order_count', 'sum'),
                                                                             delivery_duration_total_days=('customer_delivery_duration_total_days', 'sum'),
                                                                             reviewed_order_count=('customer_reviewed_order_count', 'sum'),
                                                                             review_score_total=('customer_review_score_total', 'sum'),
                                                                             review_response_over_30_days_order_count=('customer_review_response_over_30_days_order_count', 'sum'),
                                                                             average_customer_relationship_duration_days=('customer_relationship_duration_days', 'mean'),
                                                                             average_customer_recency_days=('customer_recency_days', 'mean')
                                                                             ).reset_index()

customer_type_comparison

,customer_type,customer_count,total_order_count,total_item_count,total_gmv,total_freight_value,valid_delivery_order_count,late_order_count,delivery_duration_total_days,reviewed_order_count,review_score_total,review_response_over_30_days_order_count,average_customer_relationship_duration_days,average_customer_recency_days
0,one_time,90557,90557,102984,12493089.36,2062496.93,90549,6195,1.138612e+06,89963,373529.000000,611,0.000000,238.013508
1,repeat,2801,5921,7213,728408.75,135778.71,5921,339,7.287875e+04,5869,24766.666667,34,88.164526,219.810672


In [66]:
# Validate the additive customer-type comparison totals.
print("customer_type_comparison shape: ", customer_type_comparison.shape)
print("Unique customer types: ", customer_type_comparison['customer_type'].unique().tolist())
print("Duplicate customer_type values: ", customer_type_comparison['customer_type'].duplicated().sum())
print("Missing values across the table: ", customer_type_comparison.isna().sum().sum())
print("Sum of customer_count: ", customer_type_comparison['customer_count'].sum())
print("Sum of total_order_count: ", customer_type_comparison['total_order_count'].sum())
print("Sum of total_item_count: ", customer_type_comparison['total_item_count'].sum())
print("Sum of total_gmv reconciles with customer_performance['customer_gmv'].sum(): ", np.isclose(customer_type_comparison['total_gmv'].sum(), customer_performance['customer_gmv'].sum()))
print("Sum of total_freight_value reconciles with the customer freight total.: ", np.isclose(customer_type_comparison['total_freight_value'].sum(), customer_performance['customer_freight_value'].sum()))
print("Sum of valid_delivery_order_count: ", customer_type_comparison['valid_delivery_order_count'].sum())
print("Sum of late_order_count: ", customer_type_comparison['late_order_count'].sum())
print("Sum of delivery_duration_total_days reconciles with regional_order_scope['delivery_duration_days'].sum(): ", np.isclose(customer_type_comparison['delivery_duration_total_days'].sum(), regional_order_scope['delivery_duration_days'].sum()))
print("Sum of reviewed_order_count: ", customer_type_comparison['reviewed_order_count'].sum())
print("Sum of review_score_total reconciles with regional_order_scope['average_review_score'].sum(): ", np.isclose(customer_type_comparison['review_score_total'].sum(), regional_order_scope['average_review_score'].sum()))
print("Sum of review_response_over_30_days_order_count: ", customer_type_comparison['review_response_over_30_days_order_count'].sum())


customer_type_comparison shape:  (2, 14)
Unique customer types:  ['one_time', 'repeat']
Duplicate customer_type values:  0
Missing values across the table:  0
Sum of customer_count:  93358
Sum of total_order_count:  96478
Sum of total_item_count:  110197
Sum of total_gmv reconciles with customer_performance['customer_gmv'].sum():  True
Sum of total_freight_value reconciles with the customer freight total.:  True
Sum of valid_delivery_order_count:  96470
Sum of late_order_count:  6534
Sum of delivery_duration_total_days reconciles with regional_order_scope['delivery_duration_days'].sum():  True
Sum of reviewed_order_count:  95832
Sum of review_score_total reconciles with regional_order_scope['average_review_score'].sum():  True
Sum of review_response_over_30_days_order_count:  645


In [67]:
# Calculate customer-type contribution, commercial, delivery, and review metrics.
customer_type_comparison['customer_share_percentage'] = (customer_type_comparison['customer_count']/customer_type_comparison['customer_count'].sum()) * 100

customer_type_comparison['order_share_percentage'] = (customer_type_comparison['total_order_count']/customer_type_comparison['total_order_count'].sum()) * 100

customer_type_comparison['item_share_percentage'] = (customer_type_comparison['total_item_count']/customer_type_comparison['total_item_count'].sum()) * 100

customer_type_comparison['gmv_share_percentage'] = (customer_type_comparison['total_gmv']/customer_type_comparison['total_gmv'].sum()) * 100

customer_type_comparison['average_orders_per_customer'] = customer_type_comparison['total_order_count']/customer_type_comparison['customer_count']

customer_type_comparison['average_items_per_customer'] = customer_type_comparison['total_item_count']/customer_type_comparison['customer_count']

customer_type_comparison['average_gmv_per_customer'] = customer_type_comparison['total_gmv']/customer_type_comparison['customer_count']

customer_type_comparison['average_order_value'] = customer_type_comparison['total_gmv']/customer_type_comparison['total_order_count']

customer_type_comparison['average_items_per_order'] = customer_type_comparison['total_item_count']/customer_type_comparison['total_order_count']

customer_type_comparison['freight_burden_percentage'] = (customer_type_comparison['total_freight_value']/customer_type_comparison['total_gmv']) * 100

customer_type_comparison['average_delivery_duration_days'] = customer_type_comparison['delivery_duration_total_days']/customer_type_comparison['valid_delivery_order_count']

customer_type_comparison['late_delivery_rate'] = (customer_type_comparison['late_order_count']/customer_type_comparison['valid_delivery_order_count']) * 100

customer_type_comparison['review_coverage_rate'] = (customer_type_comparison['reviewed_order_count']/customer_type_comparison['total_order_count']) * 100

customer_type_comparison['average_review_score'] = customer_type_comparison['review_score_total']/ customer_type_comparison['reviewed_order_count']

customer_type_comparison['review_response_over_30_days_rate'] = (customer_type_comparison['review_response_over_30_days_order_count']/customer_type_comparison['reviewed_order_count']) * 100

In [68]:
# Validate the derived customer-type comparison metrics.
print("customer_type_comparison shape: ", customer_type_comparison.shape)

derived_columns = ['customer_share_percentage',
                   'order_share_percentage',
                   'item_share_percentage',
                   'gmv_share_percentage',
                   'average_orders_per_customer',
                   'average_items_per_customer',
                   'average_gmv_per_customer',
                   'average_order_value',
                   'average_items_per_order', 
                   'freight_burden_percentage',
                   'average_delivery_duration_days',
                   'late_delivery_rate',
                   'review_coverage_rate',
                   'average_review_score',
                   'review_response_over_30_days_rate']

print("\nderived columns data type:","\n", customer_type_comparison[derived_columns].dtypes)

print("\nMissing values across the derived columns: ", customer_type_comparison[derived_columns].isna().sum().sum())

print("\nInfinite values across the derived columns: ", np.isinf(customer_type_comparison[derived_columns]).sum().sum())

customer_type_comparison shape:  (2, 29)

derived columns data type: 
 customer_share_percentage            float64
order_share_percentage               float64
item_share_percentage                float64
gmv_share_percentage                 float64
average_orders_per_customer          float64
average_items_per_customer           float64
average_gmv_per_customer             float64
average_order_value                  float64
average_items_per_order              float64
freight_burden_percentage            float64
average_delivery_duration_days       float64
late_delivery_rate                   float64
review_coverage_rate                 float64
average_review_score                 float64
review_response_over_30_days_rate    Float64
dtype: object

Missing values across the derived columns:  0

Infinite values across the derived columns:  0


In [69]:
print("\nCustomer shares sum to approximately 100%: ", np.isclose(customer_type_comparison['customer_share_percentage'].sum(), 100))

print("\nOrder shares sum to approximately 100%: ", np.isclose(customer_type_comparison['order_share_percentage'].sum(), 100))

print("\nItem shares sum to approximately 100%: ", np.isclose(customer_type_comparison['item_share_percentage'].sum(), 100))

print("\nGMV shares sum to approximately 100%: ", np.isclose(customer_type_comparison['gmv_share_percentage'].sum(), 100))

print("\nAll freight-burden percentages are non-negative: ", (customer_type_comparison['freight_burden_percentage']>=0).all())

print("\nLate-delivery rates are between 0 and 100: ", customer_type_comparison['late_delivery_rate'].between(0, 100).all())

print("\nReview-coverage rates are between 0 and 100: ", customer_type_comparison['review_coverage_rate'].between(0, 100).all())

print("\nReview-response-over-30-days rates are between 0 and 100: ", customer_type_comparison['review_response_over_30_days_rate'].between(0, 100).all())

print("\nAverage review scores are between 1 and 5: ", customer_type_comparison['average_review_score'].between(1, 5).all())

one_time_group = customer_type_comparison[customer_type_comparison['customer_type']=='one_time']
print("\nThe one-time group’s average_orders_per_customer equals 1: ", one_time_group['average_orders_per_customer'].eq(1).all())

repeat_group = customer_type_comparison[customer_type_comparison['customer_type']=='repeat']
print("\nThe repeat group’s average_orders_per_customer is greater than 1: ", repeat_group['average_orders_per_customer'].gt(1).all())

print("\nThe one-time group’s average_customer_relationship_duration_days equals 0: ", one_time_group['average_customer_relationship_duration_days'].eq(0).all())


Customer shares sum to approximately 100%:  True

Order shares sum to approximately 100%:  True

Item shares sum to approximately 100%:  True

GMV shares sum to approximately 100%:  True

All freight-burden percentages are non-negative:  True

Late-delivery rates are between 0 and 100:  True

Review-coverage rates are between 0 and 100:  True

Review-response-over-30-days rates are between 0 and 100:  True

Average review scores are between 1 and 5:  True

The one-time group’s average_orders_per_customer equals 1:  True

The repeat group’s average_orders_per_customer is greater than 1:  True

The one-time group’s average_customer_relationship_duration_days equals 0:  True


In [70]:
# Compare customer volume and commercial performance by customer type.
customer_type_commercial_columns = [
    'customer_type',
    'customer_count',
    'customer_share_percentage',
    'total_order_count',
    'order_share_percentage',
    'total_item_count',
    'item_share_percentage',
    'total_gmv',
    'gmv_share_percentage',
    'average_orders_per_customer',
    'average_items_per_customer',
    'average_gmv_per_customer',
    'average_order_value',
    'average_items_per_order',
    'freight_burden_percentage'
]

customer_type_comparison[customer_type_commercial_columns]

,customer_type,customer_count,customer_share_percentage,total_order_count,order_share_percentage,total_item_count,item_share_percentage,total_gmv,gmv_share_percentage,average_orders_per_customer,average_items_per_customer,average_gmv_per_customer,average_order_value,average_items_per_order,freight_burden_percentage
0,one_time,90557,96.999722,90557,93.86285,102984,93.45445,12493089.36,94.490725,1.000000,1.137228,137.958295,137.958295,1.137228,16.509103
1,repeat,2801,3.000278,5921,6.13715,7213,6.54555,728408.75,5.509275,2.113888,2.575152,260.053106,123.021238,1.218206,18.640456


In [71]:
# Compare delivery, review, relationship-duration, and recency measures by customer type.
customer_type_experience_columns = [
    'customer_type',
    'valid_delivery_order_count',
    'average_delivery_duration_days',
    'late_delivery_rate',
    'reviewed_order_count',
    'review_coverage_rate',
    'average_review_score',
    'review_response_over_30_days_rate',
    'average_customer_relationship_duration_days',
    'average_customer_recency_days'
]

customer_type_comparison[customer_type_experience_columns]

,customer_type,valid_delivery_order_count,average_delivery_duration_days,late_delivery_rate,reviewed_order_count,review_coverage_rate,average_review_score,review_response_over_30_days_rate,average_customer_relationship_duration_days,average_customer_recency_days
0,one_time,90549,12.574545,6.841600,89963,99.34406,4.152029,0.679168,0.000000,238.013508
1,repeat,5921,12.308521,5.725384,5869,99.12177,4.219913,0.579315,88.164526,219.810672


### One-Time and Repeat Customer Comparison Summary

One-time customers account for approximately 97.00% of the customer base and 94.49% of delivered-order GMV, while repeat customers represent only 3.00% of customers but contribute 5.51% of GMV. Repeat customers generate substantially more GMV per customer—approximately R$260.05 compared with R$137.96—because they purchase more frequently and buy more items over their customer relationship.

Repeat customers have a lower average order value than one-time customers, indicating that their higher customer value is driven by purchase frequency rather than larger individual orders. They also show a slightly higher items-per-order measure and freight burden.

Customer-experience measures are modestly stronger for repeat customers: they have a lower late-delivery rate, a slightly shorter average delivery duration, a higher average review score, and a lower delayed-review-response rate. They are also more recent on average and maintain an average observed relationship duration of approximately 88 days.

These results identify customer retention as a major commercial opportunity because only a small proportion of customers make repeat delivered purchases. The comparison is observational and does not establish that delivery or review performance directly causes repeat purchasing.

## 6. Validate and Save Regional and Customer Tables

The final regional, customer, and customer-type tables are reconciled against the analysis-ready delivered-order dataset before export. Additive order, item, GMV, freight, delivery, and review measures are checked to confirm that no records or commercial values were duplicated or omitted during aggregation.

In [72]:
# Reconcile delivered-order, delivered-item, GMV, and freight totals across tables.

master_delivered_order_count = regional_order_scope['order_id'].count()
master_delivered_item_count = delivered_order_items['order_item_id'].count()
master_gmv = delivered_order_items['price'].sum()
master_freight_value = delivered_order_items['freight_value'].sum()

commercial_reconciliation = pd.DataFrame(
    {
        'table_name': [
            'analysis_ready_master',
            'regional_performance',
            'customer_performance',
            'customer_type_comparison'
        ],
        'delivered_order_count': [
            master_delivered_order_count,
            regional_performance['regional_order_count'].sum(),
            customer_performance['customer_order_count'].sum(),
            customer_type_comparison['total_order_count'].sum()
        ],
        'delivered_item_count': [
            master_delivered_item_count,
            regional_performance['regional_item_count'].sum(),
            customer_performance['customer_item_count'].sum(),
            customer_type_comparison['total_item_count'].sum()
        ],
        'gmv': [
            master_gmv,
            regional_performance['regional_gmv'].sum(),
            customer_performance['customer_gmv'].sum(),
            customer_type_comparison['total_gmv'].sum()
        ],
        'freight_value': [
            master_freight_value,
            regional_performance['regional_freight_value'].sum(),
            customer_performance['customer_freight_value'].sum(),
            customer_type_comparison['total_freight_value'].sum()
        ]
    }
)

commercial_reconciliation 

,table_name,delivered_order_count,delivered_item_count,gmv,freight_value
0,analysis_ready_master,96478,110197,13221498.11,2198275.64
1,regional_performance,96478,110197,13221498.11,2198275.64
2,customer_performance,96478,110197,13221498.11,2198275.64
3,customer_type_comparison,96478,110197,13221498.11,2198275.64


In [73]:
# Validate the commercial reconciliation table.
print("All delivered-order totals reconcile:", commercial_reconciliation['delivered_order_count'].eq(master_delivered_order_count).all())

print("All delivered-item totals reconcile:", commercial_reconciliation['delivered_item_count'].eq(master_delivered_item_count).all())

print("All GMV totals reconcile:", np.isclose(commercial_reconciliation['gmv'], master_gmv).all())

print("All freight-value totals reconcile:", np.isclose(commercial_reconciliation['freight_value'], master_freight_value).all())

All delivered-order totals reconcile: True
All delivered-item totals reconcile: True
All GMV totals reconcile: True
All freight-value totals reconcile: True


In [74]:
# Reconcile delivery and review totals across tables.
master_valid_delivery_order_count = regional_order_scope['valid_delivery_order'].sum()
master_late_delivery_order_count = regional_order_scope['late_delivery_order'].sum()
master_delivery_duration_total_days = regional_order_scope['delivery_duration_days'].sum()
master_reviewed_order_count = regional_order_scope['reviewed_order'].sum()
master_review_score_total = regional_order_scope['average_review_score'].sum()
master_review_response_over_30_days_order_count = regional_order_scope['flag_review_response_over_30_days'].sum()

experience_reconciliation = pd.DataFrame(
    {
        'table_name': [
            'analysis_ready_master',
            'regional_performance',
            'customer_performance',
            'customer_type_comparison'],

        'valid_delivery_order_count':[
            master_valid_delivery_order_count,
            regional_performance['regional_valid_delivery_order_count'].sum(),
            customer_performance['customer_valid_delivery_order_count'].sum(),
            customer_type_comparison['valid_delivery_order_count'].sum()
        ],

        'late_order_count':[
            master_late_delivery_order_count,
            regional_performance['regional_late_order_count'].sum(),
            customer_performance['customer_late_order_count'].sum(),
            customer_type_comparison['late_order_count'].sum()
            ],

        'delivery_duration_total_days':[
            master_delivery_duration_total_days,
            (regional_performance['regional_average_delivery_duration_days']*regional_performance['regional_valid_delivery_order_count']).sum(),
            customer_performance['customer_delivery_duration_total_days'].sum(),
            customer_type_comparison['delivery_duration_total_days'].sum()
        ],

        'reviewed_order_count':[
            master_reviewed_order_count,
            regional_performance['regional_reviewed_order_count'].sum(),
            customer_performance['customer_reviewed_order_count'].sum(),
            customer_type_comparison['reviewed_order_count'].sum()
        ],

        'review_score_total':[
            master_review_score_total,
            (regional_performance['regional_average_review_score'] * regional_performance['regional_reviewed_order_count']).sum(),
            customer_performance['customer_review_score_total'].sum(),
            customer_type_comparison['review_score_total'].sum()
        ],

        'review_response_over_30_days_order_count':[
            master_review_response_over_30_days_order_count,
            regional_performance['regional_review_response_over_30_days_order_count'].sum(),
            customer_performance['customer_review_response_over_30_days_order_count'].sum(),
            customer_type_comparison['review_response_over_30_days_order_count'].sum()
        ]

    }
)

experience_reconciliation

,table_name,valid_delivery_order_count,late_order_count,delivery_duration_total_days,reviewed_order_count,review_score_total,review_response_over_30_days_order_count
0,analysis_ready_master,96470,6534,1.211491e+06,95832,398295.666667,645
1,regional_performance,96470,6534,1.211491e+06,95832,398295.666667,645
2,customer_performance,96470,6534,1.211491e+06,95832,398295.666667,645
3,customer_type_comparison,96470,6534,1.211491e+06,95832,398295.666667,645


In [75]:
# Validate the delivery and review reconciliation table.
print("Table shape:", experience_reconciliation.shape)

print("Duplicate table_name values:", experience_reconciliation['table_name'].duplicated().sum())

print("Missing values across the table:", experience_reconciliation.isna().sum().sum())

print("All valid-delivery-order totals reconcile:", experience_reconciliation['valid_delivery_order_count'].eq(master_valid_delivery_order_count).all())

print("All late-order totals reconcile:", experience_reconciliation['late_order_count'].eq(master_late_delivery_order_count).all())

print("All delivery-duration totals reconcile:", np.isclose(experience_reconciliation['delivery_duration_total_days'], master_delivery_duration_total_days).all())

print("All reviewed-order totals reconcile:", experience_reconciliation['reviewed_order_count'].eq(master_reviewed_order_count).all())

print("All review-score totals reconcile:", np.isclose(experience_reconciliation['review_score_total'], master_review_score_total).all())

print("All response-over-30-days totals reconcile:", experience_reconciliation['review_response_over_30_days_order_count'].eq(master_review_response_over_30_days_order_count).all())

Table shape: (4, 7)
Duplicate table_name values: 0
Missing values across the table: 0
All valid-delivery-order totals reconcile: True
All late-order totals reconcile: True
All delivery-duration totals reconcile: True
All reviewed-order totals reconcile: True
All review-score totals reconcile: True
All response-over-30-days totals reconcile: True


### Cross-Table Reconciliation Summary

Delivered-order, delivered-item, GMV, freight, delivery, and review totals reconcile successfully across the analysis-ready master scope, regional-performance table, customer-performance table, and customer-type comparison table. Weighted delivery-duration and review-score totals also reconcile across all analytical grains.

These checks confirm that the aggregations preserve the complete delivered-order scope without duplicating or omitting additive business and customer-experience measures. The regional and customer tables are therefore ready for export and downstream reporting.

In [76]:
# Save the validated regional and customer analysis tables.
# Identify the project root regardless of whether the notebook is launched
# from the project folder or the notebooks folder.
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Create the output directory if it does not already exist.
output_directory = project_root/"reports"/"tables"
output_directory.mkdir(parents=True, exist_ok=True)

# Map each output filename to its validated DataFrame.
tables_to_export = {
    "regional_performance.csv": regional_performance,
    "low_performance_regional_candidates.csv":
        low_performance_regional_candidates,
    "customer_performance.csv": customer_performance,
    "customer_type_comparison.csv": customer_type_comparison,
    "commercial_reconciliation.csv": commercial_reconciliation,
    "experience_reconciliation.csv": experience_reconciliation
}

# Export every table without the pandas index.
for file_name, dataframe in tables_to_export.items():
    dataframe.to_csv(output_directory/file_name, index=False)

print("Validated regional and customer analysis tables saved to:", output_directory.resolve())

print("\nExported files:")

for file_name in tables_to_export:
    print("-", file_name)

Validated regional and customer analysis tables saved to: C:\Users\Hp\DataAnalyst\reports\tables

Exported files:
- regional_performance.csv
- low_performance_regional_candidates.csv
- customer_performance.csv
- customer_type_comparison.csv
- commercial_reconciliation.csv
- experience_reconciliation.csv


In [77]:
# Read back the exported regional and customer tables for validation.
regional_performance_export_check = pd.read_csv(output_directory/"regional_performance.csv")

low_performance_regional_candidates_export_check = pd.read_csv(output_directory/"low_performance_regional_candidates.csv")

customer_performance_export_check = pd.read_csv(output_directory/"customer_performance.csv")

customer_type_comparison_export_check = pd.read_csv(output_directory/"customer_type_comparison.csv")

commercial_reconciliation_export_check = pd.read_csv(output_directory/"commercial_reconciliation.csv")

experience_reconciliation_export_check = pd.read_csv(output_directory/"experience_reconciliation.csv")

# Display the shape of each imported table.
print("Regional performance shape:", regional_performance_export_check.shape)

print("Low-performance regional candidates shape:", low_performance_regional_candidates_export_check.shape)

print("Customer performance shape:", customer_performance_export_check.shape)

print("Customer-type comparison shape:", customer_type_comparison_export_check.shape)

print("Commercial reconciliation shape:", commercial_reconciliation_export_check.shape)

print("Experience reconciliation shape:", experience_reconciliation_export_check.shape)

Regional performance shape: (27, 26)
Low-performance regional candidates shape: (7, 26)
Customer performance shape: (93358, 27)
Customer-type comparison shape: (2, 29)
Commercial reconciliation shape: (4, 5)
Experience reconciliation shape: (4, 7)


In [78]:
# Confirm that exported tables preserve their original shapes and column structures.
export_validation_pairs = {
    "Regional performance": (
        regional_performance,
        regional_performance_export_check
    ),
    "Low-performance regional candidates": (
        low_performance_regional_candidates,
        low_performance_regional_candidates_export_check
    ),
    "Customer performance": (
        customer_performance,
        customer_performance_export_check
    ),
    "Customer-type comparison": (
        customer_type_comparison,
        customer_type_comparison_export_check
    ),
    "Commercial reconciliation": (
        commercial_reconciliation,
        commercial_reconciliation_export_check
    ),
    "Experience reconciliation": (
        experience_reconciliation,
        experience_reconciliation_export_check
    )
}

for table_name, (original_table, exported_table) in export_validation_pairs.items():
    print(f"{table_name} shape preserved:",
          exported_table.shape == original_table.shape)

    print(f"{table_name} columns preserved:",
          exported_table.columns.equals(original_table.columns))

    print()

Regional performance shape preserved: True
Regional performance columns preserved: True

Low-performance regional candidates shape preserved: True
Low-performance regional candidates columns preserved: True

Customer performance shape preserved: True
Customer performance columns preserved: True

Customer-type comparison shape preserved: True
Customer-type comparison columns preserved: True

Commercial reconciliation shape preserved: True
Commercial reconciliation columns preserved: True

Experience reconciliation shape preserved: True
Experience reconciliation columns preserved: True



### Export Validation Summary

All six regional and customer analysis tables were exported successfully and read back from the reporting directory. The reloaded tables preserve the expected row and column counts as well as their original column structures.

This confirms that the regional-performance, low-performance candidate, customer-performance, customer-type comparison, and reconciliation tables are ready for downstream reporting and visualization.

In [79]:
# Inspect regional and customer-type summary columns before SQL–Python reconciliation.
print("regional_performance shape: ", regional_performance.shape)
print("regional_performance columns: ", regional_performance.columns)
print("customer_type_comparison shape: ", customer_type_comparison.shape)
print("customer_type_comparison columns: ", customer_type_comparison.columns)

regional_performance shape:  (27, 26)
regional_performance columns:  Index(['customer_state', 'regional_item_count', 'regional_gmv',
       'regional_freight_value', 'regional_order_count',
       'regional_customer_count', 'regional_average_delivery_duration_days',
       'regional_valid_delivery_order_count', 'regional_late_order_count',
       'regional_reviewed_order_count', 'regional_average_review_score',
       'regional_review_response_over_30_days_order_count',
       'regional_average_order_value', 'regional_average_items_per_order',
       'regional_order_share_percentage', 'regional_item_share_percentage',
       'regional_gmv_share_percentage', 'regional_freight_burden_percentage',
       'regional_late_delivery_rate', 'regional_review_coverage_rate',
       'regional_review_response_over_30_days_rate',
       'regional_order_count_rank', 'regional_gmv_rank',
       'low_regional_order_volume_candidate', 'low_regional_gmv_candidate',
       'low_regional_performance_candid

In [80]:
# Display every customer-type summary column without truncation.
for column_name in customer_type_comparison.columns:
    print(column_name)

customer_type
customer_count
total_order_count
total_item_count
total_gmv
total_freight_value
valid_delivery_order_count
late_order_count
delivery_duration_total_days
reviewed_order_count
review_score_total
review_response_over_30_days_order_count
average_customer_relationship_duration_days
average_customer_recency_days
customer_share_percentage
order_share_percentage
item_share_percentage
gmv_share_percentage
average_orders_per_customer
average_items_per_customer
average_gmv_per_customer
average_order_value
average_items_per_order
freight_burden_percentage
average_delivery_duration_days
late_delivery_rate
review_coverage_rate
average_review_score
review_response_over_30_days_rate


In [81]:
# Display customer-type counts, merchandise GMV, and percentage shares for SQL comparison.
desired_columns_customers_type   = ['customer_type',
                   'customer_count',
                   'total_order_count',
                   'total_item_count',
                   'total_gmv',
                   'customer_share_percentage',
                   'order_share_percentage',
                   'gmv_share_percentage']

customer_type_comparison[desired_columns_customers_type].sort_values('customer_type')

,customer_type,customer_count,total_order_count,total_item_count,total_gmv,customer_share_percentage,order_share_percentage,gmv_share_percentage
0,one_time,90557,90557,102984,12493089.36,96.999722,93.86285,94.490725
1,repeat,2801,5921,7213,728408.75,3.000278,6.13715,5.509275


In [88]:
# Display state-level commercial measures for SQL–Python reconciliation.
desired_column_regional_performance = ['customer_state',
                                       'regional_order_count',
                                       'regional_item_count',
                                       'regional_gmv',
                                       'regional_average_order_value',
                                       'regional_gmv_share_percentage']

regional_performance[desired_column_regional_performance].sort_values(by=['regional_gmv','customer_state'], ascending=[False, True])

,customer_state,regional_order_count,regional_item_count,regional_gmv,regional_average_order_value,regional_gmv_share_percentage
25,SP,40501,46448,5067633.16,125.123655,38.328736
18,RJ,12350,14143,1759651.13,142.481873,13.309015
10,MG,11354,12916,1552481.83,136.734352,11.742102
22,RS,5345,6134,728897.47,136.369966,5.512972
17,PR,4923,5649,666063.51,135.296264,5.037731
23,SC,3546,4097,507012.13,142.981424,3.834756
4,BA,3256,3683,493584.14,151.592181,3.733194
6,DF,2080,2355,296498.41,142.547312,2.242548
8,GO,1957,2277,282836.70,144.525652,2.139218
7,ES,1995,2225,268643.45,134.658371,2.031868


In [83]:
# Load the SQL state-performance export and inspect its structure before comparison.
sql_state_performance = pd.read_csv("../reports/tables/sql_state_performance.csv", header=0)

# Inspect the export's dimensions, column names, and first five rows.
print("Shape:", sql_state_performance.shape)
print("Columns:", sql_state_performance.columns.tolist())
display(sql_state_performance.head())

Shape: (27, 6)
Columns: ['customer_state', 'delivered_order_count', 'delivered_item_count', 'merchandise_gmv', 'average_order_merchandise_value', 'gmv_share_pct']


,customer_state,delivered_order_count,delivered_item_count,merchandise_gmv,average_order_merchandise_value,gmv_share_pct
0,SP,40501,46448,5067633.16,125.123655,38.328736
1,RJ,12350,14143,1759651.13,142.481872,13.309014
2,MG,11354,12916,1552481.83,136.734351,11.742102
3,RS,5345,6134,728897.47,136.369966,5.512971
4,PR,4923,5649,666063.51,135.296264,5.037731


In [90]:
# Match SQL and Python state summaries for reconciliation, retaining unmatched states.
state_comparison = sql_state_performance.merge(regional_performance[desired_column_regional_performance], on='customer_state', how='outer', validate='one_to_one', indicator=True)

print('state_comparison size: ', state_comparison.shape)
print('state_comparison duplicated count: ', state_comparison['customer_state'].duplicated().sum())
print('state_comparison merge indicator count: ', state_comparison['_merge'].value_counts())

state_comparison size:  (27, 12)
state_comparison duplicated count:  0
state_comparison merge indicator count:  _merge
both          27
left_only      0
right_only     0
Name: count, dtype: int64


In [91]:
state_comparison

,customer_state,delivered_order_count,delivered_item_count,merchandise_gmv,average_order_merchandise_value,gmv_share_pct,regional_order_count,regional_item_count,regional_gmv,regional_average_order_value,regional_gmv_share_percentage,_merge
0,AC,80,91,15930.97,199.137125,0.120492,80,91,15930.97,199.137125,0.120493,both
1,AL,397,427,78855.72,198.629017,0.596420,397,427,78855.72,198.629018,0.596420,both
2,AM,145,163,22155.84,152.798896,0.167574,145,163,22155.84,152.798897,0.167574,both
3,AP,67,81,13374.81,199.624029,0.101159,67,81,13374.81,199.624030,0.101160,both
4,BA,3256,3683,493584.14,151.592180,3.733193,3256,3683,493584.14,151.592181,3.733194,both
5,CE,1279,1426,219757.38,171.819687,1.662121,1279,1426,219757.38,171.819687,1.662122,both
6,DF,2080,2355,296498.41,142.547312,2.242547,2080,2355,296498.41,142.547312,2.242548,both
7,ES,1995,2225,268643.45,134.658370,2.031868,1995,2225,268643.45,134.658371,2.031868,both
8,GO,1957,2277,282836.70,144.525651,2.139218,1957,2277,282836.70,144.525652,2.139218,both
9,MA,717,800,117009.38,163.192998,0.884993,717,800,117009.38,163.192999,0.884993,both


In [107]:
# Compare state-level counts, merchandise GMV, average order values, and GMV shares.
# Create a validation summary for state-level SQL–Python comparisons.
state_validation_results = [
    {
        'SQL column': 'delivered_order_count',
        'Python column': 'regional_order_count',
        'Comparison':('Exactly equal'
                      if state_comparison['delivered_order_count'].eq(state_comparison['regional_order_count']).all()
                      else 'Not equal')
    },
    {
        'SQL column': 'delivered_item_count',
                'Python column': 'regional_item_count',
                'Comparison': ('Exactly equal'
                      if state_comparison['delivered_item_count'].eq(state_comparison['regional_item_count']).all()
                      else 'Not equal')
    },
    {
            'SQL column': 'merchandise_gmv',
                    'Python column': 'regional_gmv',
                    'Comparison': ('Absolute difference below 0.01'
                          if (state_comparison['merchandise_gmv'] - state_comparison['regional_gmv']).abs().lt(0.01).all()
                          else 'Outside tolerance')
    },
    {
            'SQL column': 'average_order_merchandise_value',
                    'Python column': 'regional_average_order_value',
                    'Comparison': ('Absolute difference below 0.01'
                          if (state_comparison['average_order_merchandise_value'] - state_comparison['regional_average_order_value']).abs().lt(0.01).all()
                          else 'Outside tolerance')
    },
    {
            'SQL column': 'gmv_share_pct',
                    'Python column': 'regional_gmv_share_percentage',
                    'Comparison': ('Absolute difference below 0.00001'
                          if (state_comparison['gmv_share_pct'] - state_comparison['regional_gmv_share_percentage']).abs().lt(0.00001).all()
                          else 'Outside tolerance')
    }

]

# Convert the validation records into a DataFrame and display it.
state_validation_summary = pd.DataFrame(state_validation_results)
display(state_validation_summary)

,SQL column,Python column,Comparison
0,delivered_order_count,regional_order_count,Exactly equal
1,delivered_item_count,regional_item_count,Exactly equal
2,merchandise_gmv,regional_gmv,Absolute difference below 0.01
3,average_order_merchandise_value,regional_average_order_value,Absolute difference below 0.01
4,gmv_share_pct,regional_gmv_share_percentage,Absolute difference below 0.00001


### SQL–Python Regional Reconciliation Summary

All 27 customer states matched, with no duplicate or unmatched states.

- Delivered-order and item counts matched exactly.
- Merchandise GMV and average order values differed by less than R$0.01.
- GMV shares differed by less than 0.00001 percentage points.